# 1. ECG preprocessing and quality assessment


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, iirnotch, welch, find_peaks
from scipy.stats import wilcoxon
from skimage.metrics import structural_similarity as ssim

# =========================
# USER SETTINGS
# =========================
folder_path = "/content/drive/MyDrive/ECG New/Data_new/Raw"
out_dir     = "/content/drive/MyDrive/ECG New/Data_new"

fs = 1000
bp_low, bp_high = 0.5, 40.0
notch_freq, notch_Q = 50.0, 30
pre_ms, post_ms = 300, 400
min_rr_ms = 300
prominence_factor = 3.0
template_source = "filtered"

# Annotation position factors
annot_xpos = 1.6
annot_yfactor = 0.7

# Colors
raw_color  = "red"   # red
filt_color = "blue"   # blue

plt.rcParams.update({
    'font.size': 10,
    'font.family': 'Times New Roman',
    'axes.titlesize': 10,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'axes.linewidth': 0.9,
    'lines.linewidth': 0.9,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
})

# =========================
# Filtering Functions
# =========================
def bandpass_filter(x, low, high, fs, order=4):
    ny = 0.5 * fs
    b, a = butter(order, [low/ny, high/ny], btype='band')
    return filtfilt(b, a, x)

def notch_filter(x, f0, fs, Q=30):
    b, a = iirnotch(f0, Q, fs)
    return filtfilt(b, a, x)

def full_filter(x, fs):
    return notch_filter(bandpass_filter(x, bp_low, bp_high, fs), notch_freq, fs, notch_Q)

# =========================
# Beat-level Metrics
# =========================
def detect_r_peaks(x, fs, min_rr_ms=300, prominence_factor=3.0):
    mad = np.median(np.abs(x - np.median(x))) + 1e-12
    prom = prominence_factor * mad
    distance = int((min_rr_ms/1000.0) * fs)
    peaks, _ = find_peaks(x, distance=distance, prominence=prom)
    return peaks

def segment_beats(x, peaks, fs, pre_ms=300, post_ms=400):
    pre = int(pre_ms/1000.0 * fs)
    post = int(post_ms/1000.0 * fs)
    segs = []
    for p in peaks:
        s, e = p - pre, p + post + 1
        if s >= 0 and e <= len(x):
            segs.append(x[s:e])
    return np.vstack(segs) if segs else np.empty((0, pre+post+1))

def mse_per_beat(segments, template):
    return np.mean((segments - template[None, :])**2, axis=1)

def mae_per_beat(segments, template):
    return np.mean(np.abs(segments - template[None, :]), axis=1)

def ssim_per_beat(segments, template):
    vals = []
    for seg in segments:
        score, _ = ssim(seg, template, data_range=seg.max() - seg.min(), full=True)
        vals.append(score)
    return np.array(vals)

def prd_per_beat(segments, template):
    return 100 * np.sqrt(np.sum((segments - template[None, :])**2, axis=1) /
                         np.sum(template[None, :]**2, axis=1))

# =========================
# Power in Frequency Band
# =========================
def band_power(freqs, psd, fmin, fmax):
    mask = (freqs >= fmin) & (freqs < fmax)
    return np.trapezoid(psd[mask], freqs[mask]) if np.any(mask) else 0.0

# =========================
# MAIN LOOP
# =========================
results = []
for file in sorted(os.listdir(folder_path)):
    if not file.endswith(".xlsx"):
        continue

    df = pd.read_excel(os.path.join(folder_path, file), skiprows=1)
    ecg_raw = df.iloc[:, 1].values.astype(float)
    ecg_filt = full_filter(ecg_raw, fs)

    # Frequency-domain metrics
    freqs_raw, psd_raw = welch(ecg_raw, fs=fs, nperseg=2048)
    freqs_filt, psd_filt = welch(ecg_filt, fs=fs, nperseg=2048)

    baseline_raw = band_power(freqs_raw, psd_raw, 0, 0.5)
    baseline_filt = band_power(freqs_filt, psd_filt, 0, 0.5)

    ecgband_raw = band_power(freqs_raw, psd_raw, 0.5, 40)
    ecgband_filt = band_power(freqs_filt, psd_filt, 0.5, 40)

    muscle_raw = band_power(freqs_raw, psd_raw, 40, 100)
    muscle_filt = band_power(freqs_filt, psd_filt, 40, 100)

    powerline_raw = band_power(freqs_raw, psd_raw, 49.5, 50.5)
    powerline_filt = band_power(freqs_filt, psd_filt, 49.5, 50.5)

    snr_raw = 10*np.log10(ecgband_raw / (baseline_raw + muscle_raw + powerline_raw + 1e-12))
    snr_filt = 10*np.log10(ecgband_filt / (baseline_filt + muscle_filt + powerline_filt + 1e-12))

    # Time-domain beat-level metrics
    peaks = detect_r_peaks(ecg_filt, fs, min_rr_ms, prominence_factor)
    seg_raw = segment_beats(ecg_raw, peaks, fs, pre_ms, post_ms)
    seg_filt = segment_beats(ecg_filt, peaks, fs, pre_ms, post_ms)
    if seg_raw.shape[0] < 5:
        continue

    template = np.median(seg_filt, axis=0) if template_source == "filtered" else np.median(seg_raw, axis=0)

    mse_raw = np.mean(mse_per_beat(seg_raw, template))
    mse_filt = np.mean(mse_per_beat(seg_filt, template))

    mae_raw = np.mean(mae_per_beat(seg_raw, template))
    mae_filt = np.mean(mae_per_beat(seg_filt, template))

    ssim_raw = np.mean(ssim_per_beat(seg_raw, template))
    ssim_filt = np.mean(ssim_per_beat(seg_filt, template))

    prd_raw = np.mean(prd_per_beat(seg_raw, template))
    prd_filt = np.mean(prd_per_beat(seg_filt, template))

    results.append({
        "File": file,
        "BaselinePower_Raw": baseline_raw, "BaselinePower_Filt": baseline_filt,
        "MusclePower_Raw": muscle_raw, "MusclePower_Filt": muscle_filt,
        "PowerlinePower_Raw": powerline_raw, "PowerlinePower_Filt": powerline_filt,
        "ECGBandPower_Raw": ecgband_raw, "ECGBandPower_Filt": ecgband_filt,
        "SNR_Raw": snr_raw, "SNR_Filt": snr_filt,
        "MSE_Raw": mse_raw, "MSE_Filt": mse_filt,
        "MAE_Raw": mae_raw, "MAE_Filt": mae_filt,
        "SSIM_Raw": ssim_raw, "SSIM_Filt": ssim_filt,
        "PRD_Raw": prd_raw, "PRD_Filt": prd_filt
    })

# =========================
# Save CSVs
# =========================
df_results = pd.DataFrame(results)
os.makedirs(out_dir, exist_ok=True)
df_results.to_csv(os.path.join(out_dir, "ECG_Filtering_Performance_AllMetrics.csv"), index=False)

summary = df_results.drop(columns=['File']).agg(['mean', 'std'])
summary.to_csv(os.path.join(out_dir, "ECG_Filtering_Performance_AllMetrics_SUMMARY.csv"))

# =========================
# PLOT
# =========================
fig, axs = plt.subplots(3, 3, figsize=(9, 6))
axs = axs.ravel()

metrics_info = [
    ("BaselinePower", "Baseline Wander Power (V²/Hz)", True),
    ("MusclePower", "Muscle Noise Power (V²/Hz)", True),
    ("PowerlinePower", "Powerline Power (V²/Hz)", True),
    ("ECGBandPower", "ECG Band Power (V²/Hz)", False),
    ("SNR", "Signal-to-Noise Ratio (dB)", False),
    ("MSE", "Mean Squared Error", False),
    ("MAE", "Mean Absolute Error", False),
    ("SSIM", "Structural Similarity Index", False),
    ("PRD", "Percent Root-Mean-Square Diff. (%)", False)
]
panel_tags = [f"({chr(97+i)})" for i in range(len(metrics_info))]

for i, (prefix, ylab, logscale) in enumerate(metrics_info):
    ax = axs[i]
    raw_vals = df_results[f"{prefix}_Raw"]
    filt_vals = df_results[f"{prefix}_Filt"]

    bp = ax.boxplot([raw_vals, filt_vals], labels=["Raw", "Filtered"], patch_artist=True, widths=0.6)
    for j, box in enumerate(bp['boxes']):
        box.set_facecolor(raw_color if j == 0 else filt_color)
        box.set_edgecolor('black')

    for w in bp['whiskers']: w.set_color('black')
    for c in bp['caps']: c.set_color('black')
    for m in bp['medians']: m.set_color('black')

    ax.set_ylabel(ylab)
    ax.tick_params(axis='both', direction='in', top=True, right=True)
    if logscale:
        ax.set_yscale('log')
        ax.tick_params(axis='y', which='minor', left=False, right=False)

    # % Change Annotation
    raw_mean, filt_mean = np.mean(raw_vals), np.mean(filt_vals)
    change_pct = ((filt_mean - raw_mean) / raw_mean) * 100
    ymax = max(np.max(raw_vals), np.max(filt_vals))
    ax.text(annot_xpos, ymax * annot_yfactor, f"Δ {change_pct:.1f}%",
            ha='center', va='bottom', fontsize=9, fontweight='bold',
            color='darkgreen' if (prefix != "PRD" and change_pct > 0) or (prefix == "PRD" and change_pct < 0) else 'red')

    ax.text(0.98, 0.98, panel_tags[i], transform=ax.transAxes,
            fontsize=10, fontweight='bold', ha='right', va='top')

plt.tight_layout()
plot_path = os.path.join(out_dir, "ECG_Filtering_Performance_9Panels.pdf")
plt.savefig(plot_path, format='pdf', bbox_inches='tight', pad_inches=0.05)
plt.show()

print(f"📊 Figure saved to: {plot_path}")


In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from math import sqrt
from typing import Tuple

# =========================
# USER SETTINGS
# =========================
in_file = "/content/drive/MyDrive/ECG New/Data_new/ECG_Filtering_Performance_AllMetrics.csv"
out_file = "/content/drive/MyDrive/ECG New/Data_new/ECG_Filtering_Table1.csv"

# =========================
# FUNCTION: Cohen's d (paired) with 95% CI
# =========================
def cohens_d_paired(x, y, alpha=0.05):
    """Compute paired Cohen's d and 95% CI (approx)."""
    diff = np.array(y) - np.array(x)
    mean_diff = np.mean(diff)
    sd_diff = np.std(diff, ddof=1)
    d = mean_diff / sd_diff if sd_diff > 0 else np.nan

    # 95% CI for Cohen's d using SE approximation
    n = len(diff)
    se_d = sqrt((1/n) + (d**2 / (2*n)))
    ci_low = d - 1.96 * se_d
    ci_high = d + 1.96 * se_d
    return d, ci_low, ci_high

# =========================
# LOAD RESULTS
# =========================
df = pd.read_csv(in_file)

metrics_info = [
    ("BaselinePower", "Baseline Wander Power (V²/Hz)", "reduction"),
    ("MusclePower", "Muscle Noise Power (V²/Hz)", "reduction"),
    ("PowerlinePower", "Powerline Power (V²/Hz)", "reduction"),
    ("ECGBandPower", "ECG Band Power (V²/Hz)", "improvement"),
    ("SNR", "Signal-to-Noise Ratio (dB)", "improvement"),
    ("MSE", "Mean Squared Error", "reduction"),
    ("MAE", "Mean Absolute Error", "reduction"),
    ("SSIM", "Structural Similarity Index", "improvement"),
    ("PRD", "Percent RMS Diff. (%)", "reduction")
]

# =========================
# BUILD TABLE
# =========================
rows = []
for key, label, change_type in metrics_info:
    raw = df[f"{key}_Raw"].values
    filt = df[f"{key}_Filt"].values

    raw_mean, raw_std = np.mean(raw), np.std(raw, ddof=1)
    filt_mean, filt_std = np.mean(filt), np.std(filt, ddof=1)
    delta_mean = filt_mean - raw_mean

    # % change
    if change_type == "reduction":
        pct_change = (raw_mean - filt_mean) / (raw_mean + 1e-12) * 100
    else:
        pct_change = (filt_mean - raw_mean) / (raw_mean + 1e-12) * 100

    # Wilcoxon signed-rank
    try:
        stat, pval = wilcoxon(raw, filt)
    except ValueError:  # if all diffs are zero
        pval = 1.0

    # Cohen's d paired
    d, ci_low, ci_high = cohens_d_paired(raw, filt)

    rows.append({
        "Metric": label,
        "Raw mean±SD": f"{raw_mean:.8f} ± {raw_std:.8f}",
        "Filt mean±SD": f"{filt_mean:.8f} ± {filt_std:.8f}",
        "ΔMean": f"{delta_mean:.8f}",
        "% Change": f"{pct_change:.1f}%",
        "p-value": f"{pval:.3e}",
        "Cohen's d [95% CI]": f"{d:.2f} [{ci_low:.2f}, {ci_high:.2f}]"
    })

# =========================
# SAVE TABLE
# =========================
df_table = pd.DataFrame(rows)
df_table.to_csv(out_file, index=False)

print(f"✅ Table 1 with %Change saved to: {out_file}")
print(df_table)


# 2. ECG fiducial detection and feature extraction


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, iirnotch, find_peaks

# =========================
# USER SETTINGS
# =========================
folder_path   = "/content/drive/MyDrive/ECG New/Data_new/Raw"  # folder with 1.xlsx, 2.xlsx, ...
out_dir       = "/content/drive/MyDrive/ECG New/Data_new"
subject_file  = "27.xlsx"   # which subject to use for the representative figure

fs = 1000
bp_low, bp_high = 0.5, 40.0
notch_freq, notch_Q = 50.0, 30

# segmentation window around R for the figure & beat processing
window_pre_ms, window_post_ms = 400, 400

# plotting
font_size = 12                     # control figure font size
wave_color = "#2563eb"             # blue
fid_color  = "#dc2626"             # red
arrow_color = "#16a34a"            # green

# =========================
# FILTERS
# =========================
def bandpass_filter(x, low, high, fs, order=4):
    ny = 0.5 * fs
    b, a = butter(order, [low/ny, high/ny], btype='band')
    return filtfilt(b, a, x)

def notch_filter(x, f0, fs, Q=30):
    b, a = iirnotch(f0, Q, fs)
    return filtfilt(b, a, x)

def full_filter(x, fs):
    y = bandpass_filter(x, bp_low, bp_high, fs)
    y = notch_filter(y, notch_freq, fs, notch_Q)
    return y

# =========================
# HELPERS
# =========================
def segment_beats(x, peaks, fs, pre_ms, post_ms):
    pre = int(pre_ms/1000 * fs)
    post = int(post_ms/1000 * fs)
    segs, idxs = [], []
    for p in peaks:
        s, e = p - pre, p + post
        if s >= 0 and e <= len(x):
            segs.append(x[s:e])
            idxs.append(p)  # original index of this R-peak
    return (np.vstack(segs) if len(segs) else np.empty((0, pre+post))), np.array(idxs, dtype=int)

def choose_representative_beat(segs):
    if segs.shape[0] == 0:
        return None
    template = np.median(segs, axis=0)
    mse = np.mean((segs - template[None, :])**2, axis=1)
    return np.argmin(mse)

def detect_r_peaks(x, fs):
    # adaptive distance & prominence
    mad = np.median(np.abs(x - np.median(x))) + 1e-12
    peaks, _ = find_peaks(x, distance=int(0.6*fs), prominence=3*mad)
    return peaks

def baseline_value(seg):
    # pre-QRS baseline: first 100 samples (100 ms) of the window
    return float(np.median(seg[:100])) if len(seg) >= 100 else float(np.median(seg))

def find_q_idx(seg, r_idx):
    lo = max(r_idx - 40, 0)
    hi = min(r_idx + 10, len(seg))
    return int(np.argmin(seg[lo:hi]) + lo)

def find_s_idx(seg, r_idx):
    lo = r_idx
    hi = min(r_idx + 80, len(seg))
    return int(np.argmin(seg[lo:hi]) + lo)

def find_p_peak_idx(seg, r_idx):
    lo = max(r_idx - 200, 0)
    hi = max(r_idx - 50, lo+1)
    return int(np.argmax(seg[lo:hi]) + lo)

def find_t_peak_idx(seg, s_idx):
    lo = min(s_idx + 40, len(seg)-1)
    hi = min(s_idx + 400, len(seg))
    if hi <= lo + 1:
        return lo
    return int(np.argmax(seg[lo:hi]) + lo)

def find_t_onset_idx(seg, s_idx, t_peak_idx):
    # first significant upward slope after S on the way to T-peak
    lo = min(s_idx + 20, len(seg)-2)
    hi = max(t_peak_idx, lo+2)
    d = np.diff(seg[lo:hi])
    if len(d) == 0:
        return lo
    thr = 0.1 * np.max(d) if np.max(d) > 0 else 0
    cand = np.where(d > thr)[0]
    return int(lo + cand[0] + 1) if len(cand) else lo

def find_t_end_idx(seg, t_peak_idx, baseline):
    # return to near-baseline after T-peak
    lo = t_peak_idx
    hi = min(t_peak_idx + 200, len(seg))
    if hi <= lo + 1:
        return lo
    tol = 0.02 * (np.max(seg) - np.min(seg) + 1e-12)
    rel = np.where(np.abs(seg[lo:hi] - baseline) < tol)[0]
    return int(lo + rel[0]) if len(rel) else min(t_peak_idx + 120, len(seg)-1)

def find_p_onset_offset(seg, p_peak_idx, baseline):
    # amplitude relative threshold
    amp = seg[p_peak_idx] - baseline
    if amp <= 0:
        return None, None
    tol = 0.15 * amp
    # onset (backward)
    lo = max(p_peak_idx - 60, 0)
    rel = np.where(np.abs(seg[lo:p_peak_idx] - baseline) < tol)[0]
    p_on = int(lo + rel[-1]) if len(rel) else max(p_peak_idx - 40, 0)
    # offset (forward)
    hi = min(p_peak_idx + 60, len(seg))
    rel2 = np.where(np.abs(seg[p_peak_idx:hi] - baseline) < tol)[0]
    p_off = int(p_peak_idx + rel2[0]) if len(rel2) else min(p_peak_idx + 40, len(seg)-1)
    return p_on, p_off

def compute_beat_intervals(seg, fs):
    """
    Returns a dict of intervals in milliseconds for a single beat segment.
    Keys: P_dur_ms, PR_ms, QRS_ms, ST_ms, T_dur_ms, QT_ms
    """
    n = len(seg)
    r_idx = n//2  # by construction in our window
    base = baseline_value(seg)

    # core fiducials
    q_idx = find_q_idx(seg, r_idx)
    s_idx = find_s_idx(seg, r_idx)
    p_peak = find_p_peak_idx(seg, r_idx)
    t_peak = find_t_peak_idx(seg, s_idx)
    t_on = find_t_onset_idx(seg, s_idx, t_peak)
    t_end = find_t_end_idx(seg, t_peak, base)
    p_on, p_off = find_p_onset_offset(seg, p_peak, base)

    # intervals (ms)
    ms = 1000.0 / fs
    def dur(i1, i2):
        return (i2 - i1) * ms if (i1 is not None) and (i2 is not None) else np.nan

    P_dur = dur(p_on, p_off)
    PR    = dur(p_on, q_idx) if p_on is not None else np.nan
    QRS   = dur(q_idx, s_idx)
    ST    = dur(s_idx, t_on)
    T_dur = dur(t_on, t_end)
    QT    = dur(q_idx, t_end)

    return {
        "P_dur_ms": P_dur,
        "PR_ms": PR,
        "QRS_ms": QRS,
        "ST_ms": ST,
        "T_dur_ms": T_dur,
        "QT_ms": QT,
        # Indices for optional plotting
        "_idx": {
            "P_on": p_on, "P_peak": p_peak, "P_off": p_off,
            "Q": q_idx, "R": r_idx, "S": s_idx,
            "T_on": t_on, "T_peak": t_peak, "T_end": t_end,
            "baseline": base
        }
    }

# ---------- NEW: QTc helpers ----------
def compute_qtc_from_qt_rr(qt_ms, rr_ms):
    """
    qt_ms: QT interval in ms
    rr_ms: preceding RR in ms
    Returns QTc (ms) for 4 formulas. NaN-safe.
    """
    if not np.isfinite(qt_ms) or not np.isfinite(rr_ms) or rr_ms <= 0:
        return np.nan, np.nan, np.nan, np.nan

    rr_s = rr_ms / 1000.0
    # Bazett & Fridericia use powers of RR (s)
    qtc_b = qt_ms / np.sqrt(rr_s)
    qtc_f = qt_ms / np.cbrt(rr_s)
    # Framingham (linear): QTc = QT + 154*(1 - RR)
    qtc_fr = qt_ms + 154.0 * (1.0 - rr_s)
    # Hodges: QTc = QT + 1.75*(HR - 60), HR from RR
    hr = 60.0 / rr_s
    qtc_h = qt_ms + 1.75 * (hr - 60.0)
    return qtc_b, qtc_f, qtc_fr, qtc_h
# -------------------------------------

# =========================
# PROCESS ALL SUBJECTS
# =========================
per_subject_rows = []
files = [f for f in sorted(os.listdir(folder_path)) if f.lower().endswith(".xlsx")]

for file in files:
    try:
        df = pd.read_excel(os.path.join(folder_path, file), skiprows=1)
        ecg_raw = df.iloc[:, 1].values.astype(float)
        ecg = full_filter(ecg_raw, fs)

        # R-peaks & RR/HR
        peaks = detect_r_peaks(ecg, fs)
        if len(peaks) < 3:
            continue
        rr_ms_all = np.diff(peaks) * (1000.0 / fs)
        mean_rr = float(np.nanmean(rr_ms_all))
        hr_bpm = 60000.0 / mean_rr if mean_rr > 0 else np.nan

        # segment beats around each R
        segs, peak_idxs = segment_beats(ecg, peaks, fs, window_pre_ms, window_post_ms)
        if segs.shape[0] == 0:
            continue

        # Map each segment's R index back to its position in the peaks array
        pos_of_peak = {int(p): i for i, p in enumerate(peaks)}
        # preceding RR (per segment/beat)
        rr_prev_ms_per_seg = []
        for rp in peak_idxs:
            pos = pos_of_peak.get(int(rp), None)
            if pos is None or pos == 0:
                rr_prev_ms_per_seg.append(np.nan)
            else:
                rr_prev_ms_per_seg.append((peaks[pos] - peaks[pos-1]) * (1000.0 / fs))
        rr_prev_ms_per_seg = np.array(rr_prev_ms_per_seg, dtype=float)

        # compute intervals per beat
        beat_ivals = [compute_beat_intervals(seg, fs) for seg in segs]
        qt_ms_per_beat = np.array([b["QT_ms"] for b in beat_ivals], dtype=float)

        # ---------- NEW: QTc per beat ----------
        qtc_b_list, qtc_f_list, qtc_fr_list, qtc_h_list = [], [], [], []
        for qt_ms, rr_ms in zip(qt_ms_per_beat, rr_prev_ms_per_seg):
            qtc_b, qtc_f, qtc_fr, qtc_h = compute_qtc_from_qt_rr(qt_ms, rr_ms)
            qtc_b_list.append(qtc_b)
            qtc_f_list.append(qtc_f)
            qtc_fr_list.append(qtc_fr)
            qtc_h_list.append(qtc_h)
        qtc_b_list = np.array(qtc_b_list, dtype=float)
        qtc_f_list = np.array(qtc_f_list, dtype=float)
        qtc_fr_list = np.array(qtc_fr_list, dtype=float)
        qtc_h_list = np.array(qtc_h_list, dtype=float)
        # --------------------------------------

        # collect per-beat arrays (ignore NaNs automatically when averaging with nanmean)
        def nanmean_key(k):
            vals = np.array([b[k] for b in beat_ivals], dtype=float)
            return float(np.nanmean(vals)) if np.isfinite(vals).any() else np.nan

        subj_metrics = {
            "File": file,
            "P_dur_ms": nanmean_key("P_dur_ms"),
            "PR_ms":    nanmean_key("PR_ms"),
            "QRS_ms":   nanmean_key("QRS_ms"),
            "ST_ms":    nanmean_key("ST_ms"),
            "T_dur_ms": nanmean_key("T_dur_ms"),
            "QT_ms":    nanmean_key("QT_ms"),
            "RR_ms":    float(np.nanmean(rr_ms_all)),
            "HR_bpm":   hr_bpm,
            "Beats_Used": int(segs.shape[0]),
            # ---------- NEW: per-subject mean QTc ----------
            "QTc_Bazett_ms":     float(np.nanmean(qtc_b_list)) if np.isfinite(qtc_b_list).any() else np.nan,
            "QTc_Fridericia_ms": float(np.nanmean(qtc_f_list)) if np.isfinite(qtc_f_list).any() else np.nan,
            "QTc_Framingham_ms": float(np.nanmean(qtc_fr_list)) if np.isfinite(qtc_fr_list).any() else np.nan,
            "QTc_Hodges_ms":     float(np.nanmean(qtc_h_list)) if np.isfinite(qtc_h_list).any() else np.nan,
        }
        per_subject_rows.append(subj_metrics)

    except Exception as e:
        print(f"Skipping {file} due to error: {e}")

# save per-subject intervals
os.makedirs(out_dir, exist_ok=True)
df_intervals = pd.DataFrame(per_subject_rows)
csv_per_subject = os.path.join(out_dir, "ECG_Intervals_PerSubject.csv")
df_intervals.to_csv(csv_per_subject, index=False)
print(f"✅ Saved per-subject intervals to: {csv_per_subject}")

# summary across subjects (of the per-subject means)
cols = [
    "P_dur_ms","PR_ms","QRS_ms","ST_ms","T_dur_ms","QT_ms","RR_ms","HR_bpm","Beats_Used",
    # ---------- NEW: QTc columns ----------
    "QTc_Bazett_ms","QTc_Fridericia_ms","QTc_Framingham_ms","QTc_Hodges_ms"
]
summary = df_intervals[cols].agg(['mean','std','median'])
csv_summary = os.path.join(out_dir, "ECG_Intervals_SUMMARY.csv")
summary.to_csv(csv_summary)
print(f"✅ Saved summary (mean/std/median) to: {csv_summary}")

# =========================
# REPRESENTATIVE FIGURE (single subject)
# =========================
# load chosen subject and plot the most representative beat
df = pd.read_excel(os.path.join(folder_path, subject_file), skiprows=1)
ecg_raw = df.iloc[:, 1].values.astype(float)
ecg = full_filter(ecg_raw, fs)

peaks = detect_r_peaks(ecg, fs)
segs, peak_idxs = segment_beats(ecg, peaks, fs, window_pre_ms, window_post_ms)
if segs.shape[0] == 0:
    raise RuntimeError("No valid segments for the chosen subject.")

rep_idx = choose_representative_beat(segs)
seg = segs[rep_idx]
n = len(seg)
pre = int(window_pre_ms/1000 * fs)
post = int(window_post_ms/1000 * fs)
t_ms = np.arange(-pre, post) * (1000.0/fs)

# recompute fiducials for the representative beat
ivals = compute_beat_intervals(seg, fs)
idxs = ivals["_idx"]
base = idxs["baseline"]

# ---------- NEW: representative beat QTc (using preceding RR for that beat) ----------
# Build preceding RR for the selected beat
pos_of_peak = {int(p): i for i, p in enumerate(peaks)}
rep_peak_pos = pos_of_peak.get(int(peak_idxs[rep_idx]), None)
if rep_peak_pos is not None and rep_peak_pos > 0:
    rep_rr_ms = (peaks[rep_peak_pos] - peaks[rep_peak_pos-1]) * (1000.0 / fs)
else:
    rep_rr_ms = np.nan
rep_qt_ms = ivals["QT_ms"]
rep_qtc_b, rep_qtc_f, rep_qtc_fr, rep_qtc_h = compute_qtc_from_qt_rr(rep_qt_ms, rep_rr_ms)
# -------------------------------------------------------------

# ====== PLOT
plt.rcParams.update({'font.size': font_size, 'font.family': 'Times New Roman'})
fig, ax = plt.subplots(figsize=(5.5, 2.5))

ax.plot(t_ms, seg, color=wave_color, lw=1.5)
ax.axhline(base, color="0.5", ls="--", lw=0.8, alpha=0.6)
ax.axvline(0, color="k", ls=":", lw=0.8, alpha=0.5)

# Fiducials (points + labels)
def add_point(name, i, dy=0.08):
    if i is None:
        return
    ax.plot(t_ms[i], seg[i], "o", color=fid_color, ms=4.5)
    ax.text(t_ms[i], seg[i] + dy*(np.ptp(seg)+1e-6), name, color=fid_color,
            ha="center", va="bottom", fontsize=font_size, fontweight="bold")

add_point("P", idxs.get("P_peak"))
add_point("Q", idxs.get("Q"))
add_point("R", idxs.get("R"))
add_point("S", idxs.get("S"))
add_point("T", idxs.get("T_peak"))

ax.set_xlabel("Time (ms relative to R)")
ax.set_ylabel("Amplitude (mV)")

# ---- Ticks inside (all four sides)
ax.tick_params(direction="in", length=4, width=0.8, top=True, right=True)

# Ensure annotations aren’t clipped
pad_y = 0.35 * np.ptp(seg)
ymin = np.min(seg) - pad_y
ymax = np.max(seg) + pad_y
ax.set_ylim(ymin, ymax)
ax.margins(x=0.12)

plt.tight_layout()
pdf_path = os.path.join(out_dir, "Representative_Filtered_ECG_Fiducials.pdf")
plt.savefig(pdf_path, format="pdf", bbox_inches="tight", pad_inches=0.05)
jpg_path = os.path.join(out_dir, "Representative_Filtered_ECG_Fiducials.jpg")
plt.savefig(jpg_path, format="jpg", bbox_inches="tight", pad_inches=0.05)
plt.show()

print(f"📄 Figure saved to: {pdf_path}")

# ---------- OPTIONAL: print representative beat numbers ----------
print("Representative beat:")
print(f"  QT   = {rep_qt_ms:.1f} ms")
print(f"  RR   = {rep_rr_ms:.1f} ms")
print(f"  QTc (Bazett)     = {rep_qtc_b:.1f} ms")
print(f"  QTc (Fridericia) = {rep_qtc_f:.1f} ms")
print(f"  QTc (Framingham) = {rep_qtc_fr:.1f} ms")
print(f"  QTc (Hodges)     = {rep_qtc_h:.1f} ms")

# 3. Feature summary and correlation-based feature selection


In [ ]:
import pandas as pd
import numpy as np
import os

# Paths
folder_path = '/content/drive/MyDrive/ECG New/Data_new/Final_Filtered_data'
input_file = os.path.join(folder_path, 'advanced_features_1.xlsx')
output_file = os.path.join(folder_path, 'advanced_features_summary.xlsx')

# Load features
df = pd.read_excel(input_file)

# Select only numeric features
numeric_cols = df.select_dtypes(include=[np.number]).columns

# Compute stats
summary = pd.DataFrame({
    "Mean ± SD": df[numeric_cols].mean().round(2).astype(str) + " ± " + df[numeric_cols].std().round(2).astype(str),
    "Median [IQR]": df[numeric_cols].median().round(2).astype(str) + " [" +
                    df[numeric_cols].quantile(0.25).round(2).astype(str) + " – " +
                    df[numeric_cols].quantile(0.75).round(2).astype(str) + "]",
    "Min": df[numeric_cols].min().round(2),
    "Max": df[numeric_cols].max().round(2)
})

# Save
summary.to_excel(output_file, index=True)
print(f"Summary saved to: {output_file}")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
numeric_features = features_df.select_dtypes(include=['number'])
correlation_matrix = numeric_features.corr(method='pearson')  # Replace 'pearson' with 'spearman' if needed

# Find font path for serif
font_path = fm.findfont('Times New Roman')
font_prop = fm.FontProperties(fname=font_path, size=14)

# Create and display the heatmap
plt.figure(figsize=(16, 12))
heatmap= sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="RdBu_r", cbar=True, annot_kws={"fontproperties": font_prop})
#plt.title("Feature Correlation Heatmap", fontproperties=font_prop,  fontsize=16)

# Get the colorbar object
cbar = heatmap.collections[0].colorbar

# Update colorbar tick labels' font size
cbar.ax.tick_params(labelsize=14)  # Adjust size as needed

# Set font style for feature names (x and y axis labels)
heatmap.set_xticklabels(heatmap.get_xticklabels(), fontproperties=font_prop)  # Rotate x-axis labels for better visibility
heatmap.set_yticklabels(heatmap.get_yticklabels(), fontproperties=font_prop)


#plt.xlabel("X-axis", fontproperties=font_prop)
#plt.ylabel("Y-axis", fontproperties=font_prop)

# Save the figure
plt.savefig("/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/Image_pearson_heatmap.pdf", format="pdf", bbox_inches="tight", pad_inches=0.05)
plt.savefig("/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/Image_pearson_heatmap.jpg", format="jpg", bbox_inches="tight", pad_inches=0.05)
plt.show()



# Identify Highly Correlated Features (Correlation > 0.8 or < -0.8)
threshold = 0.9  # Define the correlation threshold
high_corr_pairs = correlation_matrix.where((correlation_matrix.abs() > threshold) & (correlation_matrix.abs() < 1.0))

# Drop rows and columns with all NaN values
high_corr_pairs_cleaned = high_corr_pairs.dropna(how='all').dropna(axis=1, how='all')

# Convert the cleaned matrix to a list of feature pairs
high_corr_list = []
for col in high_corr_pairs_cleaned.columns:
    for idx in high_corr_pairs_cleaned.index:
        value = high_corr_pairs_cleaned.at[idx, col]
        if not pd.isna(value):  # Add only non-NaN correlations
            high_corr_list.append((idx, col, value))

# Display results
print("Highly Correlated Features (Threshold > 0.9):")
if high_corr_list:
    for pair in high_corr_list:
        print(f"{pair[0]} and {pair[1]}: Correlation = {pair[2]:.2f}")
else:
    print("No highly correlated feature pairs found.")
# Save results as CSV
if high_corr_list:
    high_corr_df = pd.DataFrame(high_corr_list, columns=["Feature 1", "Feature 2", "Correlation"])
    high_corr_df.to_csv(
        "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/highly_correlated_features.csv",
        index=False
    )
    print("Correlation results saved as CSV.")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
numeric_features = features_df.select_dtypes(include=['number'])
correlation_matrix = numeric_features.corr(method='spearman')  # Replace 'pearson' with 'spearman' if needed

# Find font path for serif
font_path = fm.findfont('Times New Roman')
font_prop = fm.FontProperties(fname=font_path, size=14)

# Create and display the heatmap
plt.figure(figsize=(16, 12))
heatmap= sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="RdBu_r", cbar=True, annot_kws={"fontproperties": font_prop})
#plt.title("Feature Correlation Heatmap", fontproperties=font_prop,  fontsize=16)

# Get the colorbar object
cbar = heatmap.collections[0].colorbar

# Update colorbar tick labels' font size
cbar.ax.tick_params(labelsize=14)  # Adjust size as needed

# Set font style for feature names (x and y axis labels)
heatmap.set_xticklabels(heatmap.get_xticklabels(), fontproperties=font_prop)  # Rotate x-axis labels for better visibility
heatmap.set_yticklabels(heatmap.get_yticklabels(), fontproperties=font_prop)


#plt.xlabel("X-axis", fontproperties=font_prop)
#plt.ylabel("Y-axis", fontproperties=font_prop)

# Save the figure
plt.savefig("/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/Image_spearman_heatmap.pdf", format="pdf", bbox_inches="tight", pad_inches=0.05)
plt.savefig("/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/Image_spearman_heatmap.jpg", format="jpg", bbox_inches="tight", pad_inches=0.05)
plt.show()



# Identify Highly Correlated Features (Correlation > 0.8 or < -0.8)
threshold = 0.9  # Define the correlation threshold
high_corr_pairs = correlation_matrix.where((correlation_matrix.abs() > threshold) & (correlation_matrix.abs() < 1.0))

# Drop rows and columns with all NaN values
high_corr_pairs_cleaned = high_corr_pairs.dropna(how='all').dropna(axis=1, how='all')

# Convert the cleaned matrix to a list of feature pairs
high_corr_list = []
for col in high_corr_pairs_cleaned.columns:
    for idx in high_corr_pairs_cleaned.index:
        value = high_corr_pairs_cleaned.at[idx, col]
        if not pd.isna(value):  # Add only non-NaN correlations
            high_corr_list.append((idx, col, value))

# Display results
print("Highly Correlated Features (Threshold > 0.9):")
if high_corr_list:
    for pair in high_corr_list:
        print(f"{pair[0]} and {pair[1]}: Correlation = {pair[2]:.2f}")
else:
    print("No highly correlated feature pairs found.")
# Save results as CSV
if high_corr_list:
    high_corr_df = pd.DataFrame(high_corr_list, columns=["Feature 1", "Feature 2", "Correlation"])
    high_corr_df.to_csv(
        "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/highly_correlated_features_spearman.csv",
        index=False
    )
    print("Correlation results saved as CSV.")


# 4. Lifestyle variables and dataset merging


In [ ]:
import pandas as pd

# Load dataset
file_path = "/content/drive/MyDrive/ECG New/Lifestyle data/Lifestyle_data_bmi.csv"  # Replace with the path to your CSV file
df = pd.read_csv(file_path)

# Derived Metrics

# Calculate total MVPA (Moderate to Vigorous Physical Activity) minutes per week
df['mvpa_minutes'] = df['days_exercise'] * df['minutes_exercise']

# Categorize physical activity based on WHO guidelines (≥150 minutes per week)
df['activity_category'] = df['mvpa_minutes'].apply(lambda x: 'Active' if x >= 150 else 'Sedentary')

# Categorize strength training (≥2 days per week)
df['strength_category'] = df['strength_days'].apply(lambda x: 'Meets Guidelines' if x >= 2 else 'Does Not Meet')

# Categorize sedentary behavior
df['sedentary_category'] = df['sitting_hours'].apply(lambda x: 'Low Sedentary' if x < 6 else 'High Sedentary')

# Calculate diet score
df['diet_score'] = df['processed_meats'] + df['sugary_foods'] + df['fruits_vegetables']

# Categorize diet score (Healthy if score ≥ 12 out of 30, Unhealthy otherwise)
df['diet_category'] = df['diet_score'].apply(lambda x: 'Healthy' if x >= 12 else 'Unhealthy')

# Categorize sitting hours
def categorize_sitting_hours(hours):
    if hours == 1:
        return "Low Sedentary"
    elif hours == 2:
        return "Moderately Sedentary"
    elif hours == 3:
        return "High Sedentary"
    elif hours == 4:
        return "Very High Sedentary"
    elif hours == 5:
        return "Extremely Sedentary"

# Apply sitting hours categorization
df['Sitting_Hours_Category'] = df['sitting_hours'].apply(categorize_sitting_hours)


# 1. BMI Calculation
def calculate_bmi(Weight, Height):
    return Weight / (Height** 2)

# Apply BMI Calculation
df['BMI'] = df.apply(lambda row: calculate_bmi(row['Weight'], row['Height']), axis=1)

# BMI Categorization
def categorize_bmi(bmi):
    if bmi < 25:
        #return "Underweight"
    #elif 18.5 <= bmi < 25:
        return "Normal"
    #elif 25 <= bmi < 30:
        #return "Overweight"
    else:
        return "Overweight"

df['BMI_Category'] = df['BMI'].apply(categorize_bmi)



# Display the updated DataFrame
print(df.head())

# Save the updated dataset to a new CSV file
output_file_path = "/content/drive/MyDrive/ECG New/Lifestyle data/Lifestyle_data_with_metics_55.csv"  # Replace with desired output path
df.to_csv(output_file_path, index=False)
print(f"Updated dataset saved to {output_file_path}")


In [ ]:
import pandas as pd

# File paths
csv_path = "/content/drive/MyDrive/ECG New/Lifestyle data/Lifestyle_data_with_metics_55.csv"
excel_path = "/content/drive/MyDrive/ECG New/Data_new/Final_Filtered_data/advanced_features_1.xlsx"

# Load the files
df_csv = pd.read_csv(csv_path)
df_excel = pd.read_excel(excel_path)

# Extract the numeric part from "File Name" in excel
df_excel["Sl"] = df_excel["File Name"].str.extract(r'(\d+)').astype(int)

# Merge on "Sl"
merged_df = pd.merge(df_csv, df_excel, on="Sl", how="inner")

# --- Keep only numeric file ID ---
merged_df = merged_df.drop(columns=["File Name"])          # drop Excel's string name
merged_df = merged_df.rename(columns={"Sl": "File Name"})  # rename Sl -> File Name
merged_df = merged_df[["File Name"] + [c for c in merged_df.columns if c != "File Name"]]  # reorder

# Save merged result
out_path = "/content/drive/MyDrive/ECG New/Lifestyle data/merged_lifestyle_features_n.csv"
merged_df.to_csv(out_path, index=False)

print("Merged file saved at:", out_path)



# 5. K-means clustering and phenotype discovery


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.linear_model import LogisticRegression
from scipy.stats import ttest_ind
import seaborn as sns
import matplotlib.pyplot as plt
import math

# ==============================
# Load Dataset
# ==============================
data_path = "/content/drive/MyDrive/ECG New/Lifestyle data/merged_lifestyle_features_n.csv"
data = pd.read_csv(data_path)

# ==============================
# Define Retained Features
# ==============================

#retained_features = ['Heart Rate', 'Mean RR', 'SDNN', 'RMSSD', 'NN50', 'pNN50', 'LF Power', 'HF Power',
#'LF/HF Ratio', 'Mean Amp.', 'SD Amp.', 'RMS Amp.', 'Mean R Amp.',
#'SD R Amp.', 'R-Q Amp. Diff.', 'R-S Amp. Diff.', 'QT Int.', 'PR Int.', 'ST Elev.', 'Mean T Amp.', 'SD T Amp.', 'QRS Dur.'
retained_features = [
    'SDNN', 'RMSSD',
                     'pNN50',
                      'RMS Amp.', 'Mean R Amp.', 'R-Q Amp. Diff.',
                       'R-S Amp. Diff.',
                'QRS Dur.', "BMI",
                     "mvpa_minutes"
]

# ==============================
# Scale Features (Z-score Normalization)
# ==============================
scaler = StandardScaler()
scaled_features = scaler.fit_transform(data[retained_features])

# ==============================
# KMeans Clustering
# ==============================
kmeans = KMeans(n_clusters=2, n_init=100, random_state=42)
clusters = kmeans.fit_predict(scaled_features)

# Add cluster labels to dataset
data["Cluster"] = clusters

# Save subject → cluster mapping
output_assignments = "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/cluster_assignments.csv"
data.to_csv(output_assignments, index=False)
print(f"Cluster assignments saved to: {output_assignments}")

# ==============================
# Evaluate Clustering Performance
# ==============================
silhouette_avg = silhouette_score(scaled_features, clusters)
db_index = davies_bouldin_score(scaled_features, clusters)

print(f"Silhouette Score: {silhouette_avg:.3f}")
print(f"Davies-Bouldin Index: {db_index:.3f}")

# ==============================
# Cluster Centers
# ==============================
cluster_centers = pd.DataFrame(kmeans.cluster_centers_, columns=retained_features)
print("Cluster Centers:")
print(cluster_centers)

# Save cluster centers to CSV
output_centers = "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/cluster_centers.csv"
cluster_centers.to_csv(output_centers, index=False)
print(f"Cluster centers saved to: {output_centers}")

# ==============================
# Heatmap of Cluster Centers
# ==============================
plt.figure(figsize=(10, 8))
sns.heatmap(cluster_centers, annot=True, cmap="coolwarm", xticklabels=retained_features)
plt.title("Cluster Centers Heatmap")
plt.show()

# Transposed Heatmap for readability
cluster_centers_transposed = cluster_centers.T
plt.figure(figsize=(8, 6))
sns.heatmap(cluster_centers_transposed, annot=True, cmap='viridis', fmt=".2f",
            annot_kws={"fontsize": 14})
plt.title('Heatmap of Cluster Centers', fontdict={'family': 'Times New Roman', 'size': 14})
plt.xlabel('Cluster', fontdict={'family': 'Times New Roman', 'size': 14})
plt.ylabel('Feature', fontdict={'family': 'Times New Roman', 'size': 14})
plt.xticks(fontsize=12, family='Times New Roman')
plt.yticks(fontsize=12, family='Times New Roman')

plt.savefig('/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/heatmap_cluster_centers.pdf',
            format="pdf", bbox_inches="tight", pad_inches=0.05)
plt.show()

# ==============================
# Logistic Regression (MVPA vs Clusters)
# ==============================
# Logistic regression with both MVPA + BMI as predictors
X = data[["mvpa_minutes", "BMI"]].values
y = clusters

log_reg = LogisticRegression(random_state=42)
log_reg.fit(X, y)
predicted_clusters = log_reg.predict(X)
accuracy = (predicted_clusters == y).mean()

print(f"Logistic Regression Accuracy (MVPA + BMI predicting clusters): {accuracy:.4f}")
print("Regression Coefficients:")
print(f"MVPA: {log_reg.coef_[0][0]:.4f}, BMI: {log_reg.coef_[0][1]:.4f}, Intercept: {log_reg.intercept_[0]:.4f}")

# ==============================
# T-Tests Between Clusters
# ==============================
group1 = data.loc[clusters == 0, retained_features]
group2 = data.loc[clusters == 1, retained_features]

t_stat, p_values = ttest_ind(group1, group2, axis=0)
ttest_results = pd.DataFrame({
    "Feature": retained_features,
    "t-statistic": t_stat,
    "p-value": p_values
})

print("\nT-Test Results (Comparing Clusters on Retained Features):")
print(ttest_results)

# Save t-test results
output_ttest = "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/ttest_results.csv"
ttest_results.to_csv(output_ttest, index=False)
print(f"T-test results saved to: {output_ttest}")

# Significant features
significant_features = ttest_results[ttest_results["p-value"] < 0.05]
print("\nSignificant Features (p < 0.05):")
print(significant_features)

# ==============================
# Boxplots for Retained Features
# ==============================
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 14,
    "axes.titlesize": 15,
    "axes.labelsize": 15,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
})

num_cols = 3
num_rows = math.ceil(len(retained_features) / num_cols)

plt.figure(figsize=(14, num_rows * 5))
for i, feature in enumerate(retained_features):
    plt.subplot(num_rows, num_cols, i + 1)
    sns.boxplot(x=clusters, y=data[feature], palette="viridis")
    plt.title(feature)
    plt.xlabel("Cluster")
    plt.ylabel("Feature Value")

plt.tight_layout()
plt.savefig("/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/clustered_features_boxplots.pdf",
            format="pdf", bbox_inches="tight", pad_inches=0.05)
plt.show()


# 6. Explainable phenotype prediction (SHAP)


In [ ]:
# === SHAP for retained features (Logistic Regression) ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import shap
from matplotlib import rcParams

# -----------------------------
# Config: fonts & figure quality
# -----------------------------
#rcParams['font.family'] = 'Times New Roman'
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 12,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
})


# ==============================
# Feature rename mapping (edit as you wish)
# ==============================
feature_name_map = {
    'SDNN': 'SDNN (ms)',
    'RMSSD': 'RMSSD (ms)',
    'pNN50': 'pNN50 (%)',
    'RMS Amp.': 'RMS Amp. (mV)',
    'Mean R Amp.': 'Mean R Amp. (mV)',
    'R-Q Amp. Diff.': 'R–Q Amp. Diff. (mV)',
    'R-S Amp. Diff.': 'R–S Amp. Diff. (mV)',
    'QRS Dur.': 'QRS Dur. (ms)',
    'BMI': 'BMI (kg/m$^2$)',
    'mvpa_minutes': 'MVPA (min/Week)'
}

retained_features = list(feature_name_map.keys())
target_name = "Cluster"   # from KMeans clustering

# ==============================
# X, y
# ==============================
X = data[retained_features].copy()
y = data[target_name].values

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=42, test_size=0.25
)

# ==============================
# Logistic Regression Model
# ==============================
pipe = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=42, max_iter=5000))
])
pipe.fit(X_train, y_train)

print("Train accuracy:", pipe.score(X_train, y_train))
print("Test  accuracy:", pipe.score(X_test, y_test))

# ==============================
# SHAP Explainer
# ==============================
model_to_explain = pipe.named_steps['clf']
scaler_transform = pipe.named_steps['scaler'].transform(X)

background = shap.sample(scaler_transform, nsamples=min(100, len(scaler_transform)), random_state=42)
explainer = shap.Explainer(model_to_explain, background, feature_names=[feature_name_map[f] for f in retained_features])
shap_values = explainer(scaler_transform)

# ==============================
# Save SHAP values to CSV
# ==============================
shap_df = pd.DataFrame(shap_values.values, columns=[feature_name_map[f] for f in retained_features])
shap_df.insert(0, "Subject", data.index)
shap_df.insert(1, "Cluster", y)

output_shap = "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/shap_values.csv"
shap_df.to_csv(output_shap, index=False)
print(f"SHAP values saved to: {output_shap}")

# ==============================
# Global summary (beeswarm) with custom color
# ==============================
plt.figure(figsize=(5, 3))
shap.plots.beeswarm(shap_values, show=False, color=plt.cm.tab10)  # change colormap here
plt.tight_layout()
plt.tick_params(direction="in", length=3, width=1)
plt.savefig("/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/shap_beeswarm_retained.pdf", bbox_inches="tight", format="pdf", pad_inches=0.05)
plt.savefig("/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/shap_beeswarm_retained.jpg", bbox_inches="tight", format="jpg", pad_inches=0.05)
plt.show()

# ==============================
# Global importance (mean |SHAP|) with custom color
# ==============================
plt.figure(figsize=(5, 3.5))
shap.plots.bar(
    shap_values,
    show=False,
    max_display=len(retained_features)
      # pick any color map or RGB tuple
)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/shap_bar_retained.pdf", bbox_inches="tight", format="pdf", pad_inches=0.05)
plt.show()

# ==============================
# Odds Ratios Table
# ==============================
clf = pipe.named_steps['clf']
coef = clf.coef_.ravel()
odds = np.exp(coef)

coef_table = pd.DataFrame({
    "Feature": [feature_name_map[f] for f in retained_features],
    "Beta (std. scale)": coef,
    "Odds Ratio (exp(Beta))": odds
}).sort_values("Odds Ratio (exp(Beta))", ascending=False)

print("\nOdds Ratios Table:")
print(coef_table.to_string(index=False))


In [ ]:
# -----------------------------
# Custom global importance bar (mean |SHAP|) with color control
# -----------------------------
import matplotlib
import matplotlib.pyplot as plt
#rcParams['font.family'] = 'Times New Roman'
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 12,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
})
# mapped names (use your pretty labels)
mapped_names = [feature_name_map.get(f, f) for f in retained_features]

# ensure shap_values -> 2D array of shape (n_samples, n_features)
sv = np.array(shap_values.values)

if sv.ndim == 3:
    # common SHAP shapes: (n_samples, n_classes, n_features) or (n_samples, n_features, n_classes)
    nfeat = len(retained_features)
    if sv.shape[1] == nfeat and sv.shape[2] != nfeat:
        # (n_samples, n_features, n_classes)
        # pick class 1 if available else 0
        cls_idx = 1 if sv.shape[2] > 1 else 0
        sv = sv[:, :, cls_idx]
    elif sv.shape[2] == nfeat and sv.shape[1] != nfeat:
        # (n_samples, n_classes, n_features)
        cls_idx = 1 if sv.shape[1] > 1 else 0
        sv = sv[:, cls_idx, :]
    else:
        # fallback: flatten trailing dims (rare)
        sv = sv.reshape(sv.shape[0], -1)

elif sv.ndim == 1:
    # single-sample case
    sv = sv.reshape(1, -1)

# now sv should be shape (n_samples, n_features)
mean_abs = np.mean(np.abs(sv), axis=0)

# DataFrame for easier plotting and saving
df_mean_shap = pd.DataFrame({
    "Feature": mapped_names,
    "mean_abs_shap": mean_abs
}).sort_values("mean_abs_shap", ascending=True)

# --- Choose a colormap or a single color ---
# Option A: gradient from a colormap (inferno is warm)
cmap = matplotlib.cm.get_cmap("inferno")
colors = cmap(np.linspace(0.15, 0.85, len(df_mean_shap)))

# Option B: single warm color -> uncomment to use
# single_color = (0.85, 0.4, 0.12)  # RGB tuple
# colors = [single_color] * len(df_mean_shap)

# --- Plot ---
plt.figure(figsize=(6, max(4, 0.35 * len(df_mean_shap))))  # height scales with #features
plt.barh(df_mean_shap["Feature"], df_mean_shap["mean_abs_shap"], color=colors)
plt.xlabel("Mean |SHAP value|")
plt.title("Global feature importance (mean |SHAP|)")
plt.grid(axis="x", linestyle=":", alpha=0.4)
plt.tight_layout()
plt.tick_params(direction="in", length=3, width=1)
out_bar = "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/shap_bar_retained_custom.pdf"
plt.savefig(out_bar, bbox_inches="tight", format="pdf", pad_inches=0.05)
out_bar = "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/shap_bar_retained_custom.jpg"
plt.savefig(out_bar, bbox_inches="tight", format="jpg", pad_inches=0.05)
plt.show()

# (optional) save the summary table
out_mean_table = "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/shap_mean_abs_table.csv"
df_mean_shap.to_csv(out_mean_table, index=False)
print(f"Saved custom bar plot -> {out_bar}")
print(f"Saved mean-|SHAP| table -> {out_mean_table}")


In [ ]:
# ==============================
# Decision plot for a few subjects (fixed + fig size + subject IDs)
# ==============================
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 12,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 14,
    "ytick.labelsize": 12,
})

proba = pipe.predict_proba(X)[:, 1]
idx_sorted = np.argsort(-np.abs(proba - 0.5))[:10]  # pick top 10 most confident

# Subject IDs (from dataset index)
subject_ids = data.index[idx_sorted]

# Map feature names
mapped_names = [feature_name_map.get(f, f) for f in retained_features]

try:
    # Try new API (direct Explanation slicing)
    shap.plots.decision(shap_values[idx_sorted], feature_names=mapped_names, show=False)

    # Resize figure explicitly
    fig = plt.gcf()
    fig.set_size_inches(10, 6)

    # Add custom legend with subject IDs
    for i, sid in enumerate(subject_ids):
        plt.plot([], [], color=plt.cm.tab10(i % 10), label=f"Subject {sid}")
    plt.legend(title="Subjects", fontsize=10, title_fontsize=11, bbox_to_anchor=(1.05, 1), loc="upper left")

    fig.savefig(
        "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/shap_decision_examples.pdf",
        bbox_inches="tight", format="pdf", pad_inches=0.05
    )
    plt.close(fig)

except Exception:
    # Old API fallback
    if hasattr(shap_values, "base_values"):
        base_val = np.mean(shap_values.base_values)
    else:
        base_val = explainer.expected_value

    plt.figure(figsize=(8, 6))
    shap.decision_plot(
        base_val,
        shap_values.values[idx_sorted],
        features=X.iloc[idx_sorted].rename(columns=feature_name_map),
        feature_names=mapped_names,
        show=False
    )

    # Add custom legend with subject IDs
    for i, sid in enumerate(subject_ids):
        plt.plot([], [], color=plt.cm.tab10(i % 10), label=f"Subject {sid}")
    plt.legend(title="Subjects", fontsize=10, title_fontsize=11, bbox_to_anchor=(1.05, 1), loc="upper left")

    plt.savefig(
        "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/shap_decision_examples.pdf",
        bbox_inches="tight", format="pdf", pad_inches=0.05
    )
    plt.close()

print("Decision plot saved -> /content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/shap_decision_examples.pdf")

# -----------------------------
# Save top 10 decision plot subjects to CSV
# -----------------------------
top_subjects_df = pd.DataFrame({
    "Subject": subject_ids,
    "Cluster": y[idx_sorted],
    "Predicted_Prob_Class1": proba[idx_sorted]
})

out_csv = "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/shap_decision_top10_subjects.csv"
top_subjects_df.to_csv(out_csv, index=False)

print(f"Top 10 decision plot subjects saved -> {out_csv}")


In [ ]:
# ==============================
# Save SHAP values to CSV (with proba + base value)
# ==============================
# shap_values.values is shape (n_samples, n_features)
shap_df = pd.DataFrame(shap_values.values, columns=retained_features)

# Add identifiers
shap_df.insert(0, "Subject", data.index)     # Subject index
shap_df.insert(1, "Cluster", y)              # Cluster label

# Add model outputs
proba_class1 = pipe.predict_proba(X)[:, 1]   # probability of class 1
shap_df.insert(2, "proba_class1", proba_class1)

# Add base value (scalar or per-sample)
if hasattr(shap_values, "base_values"):
    base_val = shap_values.base_values
    # Ensure it's always 1D (broadcast scalar if needed)
    if np.ndim(base_val) == 0:
        base_val = np.repeat(base_val, len(X))
    shap_df.insert(3, "base_value", base_val)
else:
    shap_df.insert(3, "base_value", np.nan)  # fallback if missing

# Save CSV
output_shap = "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/shap_values.csv"
shap_df.to_csv(output_shap, index=False)
print(f"SHAP values (with probabilities + base values) saved to: {output_shap}")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# ==============================
# Boxplots for Retained Features (2 rows × 5 columns)
# ==============================
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 14,
    "axes.titlesize": 15,
    "axes.labelsize": 15,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
})

# Feature rename mapping (edit as you wish)
feature_name_map = {
    'SDNN': 'SDNN (ms)',
    'RMSSD': 'RMSSD (ms)',
    'pNN50': 'pNN50 (%)',
    'RMS Amp.': 'RMS Amp. (mV)',
    'Mean R Amp.': 'Mean R Amp. (mV)',
    'R-Q Amp. Diff.': 'R–Q Amp. Diff. (mV)',
    'R-S Amp. Diff.': 'R–S Amp. Diff. (mV)',
    'QRS Dur.': 'QRS Dur. (ms)',
    'BMI': 'BMI (kg/m$^2$)',
    'mvpa_minutes': 'MVPA (min/Week)'
}

num_cols = 5
num_rows = 2

# Fixed color palette: Cluster 0 = Blue, Cluster 1 = Red
palette = {0: "steelblue", 1: "crimson"}

# Subplot labels (a), (b), (c), ...
subplot_labels = [f"({chr(97+i)})" for i in range(len(retained_features))]

plt.figure(figsize=(12, 5))

for i, feature in enumerate(retained_features):
    ax = plt.subplot(num_rows, num_cols, i + 1)

    sns.boxplot(
        x="Cluster",
        y=feature,
        data=data,
        palette=palette,
        width=0.3,
        hue="Cluster",
        legend=False
    )

    # Y-axis label as feature name
    plt.ylabel(feature_name_map.get(feature, feature), fontsize=14)
    plt.xlabel("Cluster", fontsize=14)
    plt.title("")

    # Tick parameters (inside ticks)
    ax.tick_params(direction="in", length=3, width=1, top=True, right=True)

    # Add subplot label at top-right corner
    ax.text(
        0.95, 0.95, subplot_labels[i],
        transform=ax.transAxes,
        ha="right", va="top",
        fontsize=14, fontweight="bold"
    )

plt.tight_layout()
plt.savefig(
    "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/clustered_features_boxplots.pdf",
    format="pdf", bbox_inches="tight", pad_inches=0.05
)
plt.savefig(
    "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/clustered_features_boxplots.jpg",
    format="jpg", bbox_inches="tight", pad_inches=0.05
)
plt.show()


In [ ]:
# ==============================
# Heatmap of Cluster Centers with Renamed Features
# ==============================


# Feature rename mapping (edit as you wish)
feature_name_map = {
    'SDNN': 'SDNN (ms)',
    'RMSSD': 'RMSSD (ms)',
    'pNN50': 'pNN50 (%)',
    'RMS Amp.': 'RMS Amp. (mV)',
    'Mean R Amp.': 'Mean R Amp. (mV)',
    'R-Q Amp. Diff.': 'R–Q Amp. Diff. (mV)',
    'R-S Amp. Diff.': 'R–S Amp. Diff. (mV)',
    'QRS Dur.': 'QRS Dur. (ms)',
    'BMI': 'BMI (kg/m$^2$)',
    'mvpa_minutes': 'MVPA (min/Week)'
}

# Rename columns for cluster centers
cluster_centers_renamed = cluster_centers.rename(columns=feature_name_map)

# Transposed heatmap (features on y-axis)
plt.figure(figsize=(5, 4))
sns.heatmap(cluster_centers_renamed.T, annot=True, cmap="RdBu_r", fmt=".2f",
            annot_kws={"fontsize": 14})

#plt.title('Heatmap of Cluster Centers', fontdict={'family': 'Times New Roman', 'size': 14})
plt.xlabel('Cluster', fontdict={'family': 'Times New Roman', 'size': 14})
#plt.ylabel('Feature', fontdict={'family': 'Times New Roman', 'size': 14})
plt.xticks(fontsize=14, family='Times New Roman')
plt.yticks(fontsize=14, family='Times New Roman')

plt.savefig('/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/heatmap_cluster_centers_named.pdf',
            format="pdf", bbox_inches="tight", pad_inches=0.05)
plt.savefig('/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/heatmap_cluster_centers_named.jpg',
            format="jpg", bbox_inches="tight", pad_inches=0.05)
plt.show()


In [ ]:
# Save only File Name and Cluster labels
cluster_labels_only = data[["File Name"]].copy()
cluster_labels_only["Cluster"] = clusters

output_labels = "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/subject_cluster_labels.csv"
cluster_labels_only.to_csv(output_labels, index=False)

print(f"Subject cluster labels saved to: {output_labels}")


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# Function to compute Cohen's d
def cohens_d(group1, group2):
    nx, ny = len(group1), len(group2)
    dof = nx + ny - 2
    pooled_std = np.sqrt(((nx-1)*np.var(group1, ddof=1) + (ny-1)*np.var(group2, ddof=1)) / dof)
    d = (np.mean(group1) - np.mean(group2)) / pooled_std
    return d

# Function to compute 95% CI for Cohen's d using bootstrap
def cohens_d_ci(x, y, n_boot=5000, alpha=0.05):
    boot_d = []
    rng = np.random.default_rng()
    for _ in range(n_boot):
        x_resamp = rng.choice(x, size=len(x), replace=True)
        y_resamp = rng.choice(y, size=len(y), replace=True)
        boot_d.append(cohens_d(x_resamp, y_resamp))
    lower = np.percentile(boot_d, 100*alpha/2)
    upper = np.percentile(boot_d, 100*(1-alpha/2))
    return lower, upper

# Example: loop over features
features = ["SDNN", "RMSSD", "pNN50", "RMS Amp.", "Mean R Amp.", "R-Q Amp. Diff.",
            "R-S Amp. Diff.", "QRS Dur.", "BMI", "mvpa_minutes"]

results = []

for feat in features:
    # Corrected column name: 'Cluster' instead of 'cluster'
    cluster0 = data[data["Cluster"] == 0][feat].values
    cluster1 = data[data["Cluster"] == 1][feat].values

    # t-test
    t_stat, p_val = stats.ttest_ind(cluster0, cluster1, equal_var=False)

    # effect size
    d = cohens_d(cluster0, cluster1)
    ci_low, ci_high = cohens_d_ci(cluster0, cluster1)

    results.append([feat, t_stat, p_val, d, ci_low, ci_high])

# Put into a dataframe
df_results = pd.DataFrame(results, columns=["Feature", "t-stat", "p-value", "Cohen's d", "CI_low", "CI_high"])

print(df_results)

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

# Define tab10 palette
tab10 = plt.colormaps["tab10"]

# Choose cluster colors: blue (0) and red (3)
cluster_colors = {0: tab10(0), 1: tab10(3)}

plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.labelsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12
})

plt.figure(figsize=(4.5, 3))  # balanced size for journal figures

sns.scatterplot(
    x="BMI", y="mvpa_minutes",
    hue=clusters, data=data,
    palette=cluster_colors,  # ✅ tab10 blue & red
    s=90, edgecolor="k", linewidth=0.6
)

plt.xlabel("BMI (kg/m$^2$)")
plt.ylabel("MVPA (min/Week)")
plt.legend(title="Cluster", frameon=False)

# ✅ inward ticks
plt.tick_params(direction="in", length=3, width=1)

plt.tight_layout()
plt.savefig(
    "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/bmi_mvpa_scatter.pdf",
    format="pdf", bbox_inches="tight", pad_inches=0.05
)
plt.savefig(
    "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/bmi_mvpa_scatter.jpg",
    format="jpg", bbox_inches="tight", pad_inches=0.05
)
plt.show()




# 7. Supervised phenotype prediction


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.model_selection import LeaveOneOut, cross_val_score

# ===============================
# Data setup
# ===============================
X = data[["mvpa_minutes", "BMI"]].values
y = clusters

# Example placeholders (replace with your real X, y)
# X = data[["mvpa_minutes", "BMI"]].values
# y = clusters

loo = LeaveOneOut()

# ===============================
# Logistic Regression
# ===============================
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_scores = cross_val_score(log_reg, X, y, cv=loo)
log_acc = log_scores.mean()

# ===============================
# Linear Discriminant Analysis
# ===============================
lda = LinearDiscriminantAnalysis()
lda_scores = cross_val_score(lda, X, y, cv=loo)
lda_acc = lda_scores.mean()

# ===============================
# Support Vector Machine (linear kernel)
# ===============================
svm = SVC(kernel="linear", random_state=42)
svm_scores = cross_val_score(svm, X, y, cv=loo)
svm_acc = svm_scores.mean()

# ===============================
# Print results
# ===============================
print("Leave-One-Out Cross-Validation Accuracies:")
print(f"Logistic Regression: {log_acc:.4f}")
print(f"LDA:                 {lda_acc:.4f}")
print(f"SVM (linear):        {svm_acc:.4f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC

# ===============================
# Define models
# ===============================
models = {
    "LR": LogisticRegression(random_state=42, max_iter=1000),
    "LDA": LinearDiscriminantAnalysis(),
    "SVM (linear)": SVC(kernel="linear", probability=True, random_state=42)
}

# ===============================
# Fit models
# ===============================
for model in models.values():
    model.fit(X, y)

# ===============================
# Meshgrid for decision boundary
# ===============================
x_min, x_max = X[:, 0].min() - 5, X[:, 0].max() + 5
y_min, y_max = X[:, 1].min() - 2, X[:, 1].max() + 2
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 500),
    np.linspace(y_min, y_max, 500)
)

# ===============================
# Plot
# ===============================
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 14
})

fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.5))

for idx, (ax, (name, model)) in enumerate(zip(axes, models.items()), start=1):
    # Predict grid
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    # Decision boundary
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.coolwarm)

    # Scatter points
    scatter = ax.scatter(
        X[:, 0], X[:, 1], c=y,
        edgecolor="k", s=80, cmap=plt.cm.coolwarm
    )

    # Labels and title
    ax.set_xlabel("MVPA (min/week)", fontsize=14)
    ax.set_ylabel("BMI (kg/m$^2$)", fontsize=14)
    ax.set_title(name, fontsize=14)
    #ax.grid(True, linestyle="--", alpha=0.6)

    # Subplot label (a), (b), (c) in top-right
    ax.text(0.95, 0.95, f"({chr(96+idx)})",
            transform=ax.transAxes,
            fontsize=14, fontweight="bold",
            va="top", ha="right")

    # Legend inside each subplot
    ax.legend(
        *scatter.legend_elements(),
        title="Cluster", loc="upper center",
        frameon=True, fontsize=12
    )

    ax.tick_params(direction="in", length=3, width=1)

#plt.suptitle("Decision Boundaries: MVPA + BMI Predicting Clusters",
             #fontsize=16, y=1.02, family="Times New Roman")

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/decision_final.pdf',
            format="pdf", bbox_inches="tight", pad_inches=0.05)
plt.savefig('/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/decision_final.jpg',
            format="jpg", bbox_inches="tight", pad_inches=0.05)
plt.show()





In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt

# ===============================
# Evaluation: Confusion Matrix + Classification Report
# ===============================
for name, model in models.items():
    y_pred = model.predict(X)

    # Confusion Matrix
    cm = confusion_matrix(y, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Cluster 0", "Cluster 1"])
    disp.plot(cmap="Blues", values_format='d')
    plt.title(f"{name}: Confusion Matrix")
    plt.show()

    # Classification Report (dictionary)
    report = classification_report(y, y_pred, target_names=["Cluster 0", "Cluster 1"], output_dict=True)

    print(f"\n{name} - Metrics for Cluster 0 (Active):")
    print(f"Precision: {report['Cluster 0']['precision']:.4f}")
    print(f"Recall (Sensitivity): {report['Cluster 0']['recall']:.4f}")
    print(f"F1-Score: {report['Cluster 0']['f1-score']:.4f}")

    print(f"\n{name} - Metrics for Cluster 1 (Sedentary):")
    print(f"Precision: {report['Cluster 1']['precision']:.4f}")
    print(f"Recall (Sensitivity): {report['Cluster 1']['recall']:.4f}")
    print(f"F1-Score: {report['Cluster 1']['f1-score']:.4f}")

    # Full classification report (4 decimal places)
    print("\nComplete Classification Report:")
    print(classification_report(y, y_pred, target_names=["Cluster 0", "Cluster 1"], digits=4))
    print("="*60)


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 12
})

# Example: pick Logistic Regression
model = models["LR"]
y_pred = model.predict(X)

# Compute confusion matrix
cm = confusion_matrix(y, y_pred)

# Plot with custom size
fig, ax = plt.subplots(figsize=(5, 4))  # adjust (width, height) as needed
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Cluster 0", "Cluster 1"])
disp.plot(cmap="Blues", values_format='d', ax=ax, colorbar=False)
ax.tick_params(axis="both", direction="in", length=3, width=1, labelsize=12)
ax.set_title("Confusion Matrix", fontsize=12)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/confusion_matrix.pdf',
            format="pdf", bbox_inches="tight", pad_inches=0.05)
plt.savefig('/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/confusion_matrix.jpg',
            format="jpg", bbox_inches="tight", pad_inches=0.05)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# ===============================
# ROC–AUC Subplots
# ===============================
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 14
})

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))

for idx, (ax, (name, model)) in enumerate(zip(axes, models.items()), start=1):
    # Predict probabilities or decision scores
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X)[:, 1]
    else:
        y_score = model.decision_function(X)

    # ROC curve
    fpr, tpr, _ = roc_curve(y, y_score)
    roc_auc = auc(fpr, tpr)

    # Plot ROC curve
    ax.plot(fpr, tpr, color="darkblue", lw=2, label=f"AUC = {roc_auc:.2f}")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", lw=1.2)

    # Labels
    ax.set_xlabel("False Positive Rate", fontsize=14)
    ax.set_ylabel("True Positive Rate", fontsize=14)
    ax.set_title(name, fontsize=14)
    ax.legend(loc="lower right", fontsize=14, frameon=True)
    #ax.grid(alpha=0.3, linestyle="--")

    # Subplot label (a), (b), (c)
    ax.text(0.98, 0.92, f"({chr(96+idx)})",
            transform=ax.transAxes,
            fontsize=14, fontweight="bold",
            va="top", ha="right")
    ax.tick_params(direction="in", length=3, width=1)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/auc.pdf',
            format="pdf", bbox_inches="tight", pad_inches=0.05)
plt.savefig('/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/auc.jpg',
            format="jpg", bbox_inches="tight", pad_inches=0.05)
plt.show()


# 8. Cluster visualization


In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 12
})

# Assume: `X_scaled` is your scaled feature matrix
#         `kmeans` is your fitted KMeans model
palette = {0: plt.cm.tab10(0), 1: plt.cm.tab10(3)}
pca = PCA(n_components=2)
X_pca = pca.fit_transform(scaled_features)

plt.figure(figsize=(5, 3.5))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=kmeans.labels_, palette=palette, s=80, edgecolor="k")
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            c='black', s=90, alpha=0.75, marker='X', label='Centers')
#plt.title('K-Means Clusters (PCA-reduced)')
plt.xlabel("PCA Component 1", fontsize=12)
plt.ylabel("PCA Component 2", fontsize=12)
plt.legend(fontsize=10, title_fontsize=11, loc="best", frameon=True)
plt.tick_params(direction="in", length=3, width=1)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/kmeans_clusters_pca_with_centers.pdf",
            format="pdf", bbox_inches="tight", pad_inches=0.05)
plt.savefig("/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/kmeans_clusters_pca_with_centers.jpg",
            format="jpg", bbox_inches="tight", pad_inches=0.05)
plt.show()


In [ ]:
from sklearn.manifold import TSNE
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 12
})

palette = {0: plt.cm.tab10(0), 1: plt.cm.tab10(3)}
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
tsne_result = tsne.fit_transform(scaled_features)

plt.figure(figsize=(4, 3))
sns.scatterplot(x=tsne_result[:, 0], y=tsne_result[:, 1], hue=clusters, palette=palette, s=80, edgecolor="k")
plt.xlabel("t-SNE Dimension 1", fontsize=12)
plt.ylabel("t-SNE Dimension 2", fontsize=12)
#plt.title("KMeans Clusters Visualized with t-SNE")
#plt.grid(True)
plt.tick_params(direction="in", length=3, width=1)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/kmeans_clusters_tsne.pdf",
            format="pdf", bbox_inches="tight", pad_inches=0.05)
plt.savefig("/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/kmeans_clusters_pca_with_centers.jpg",
            format="jpg", bbox_inches="tight", pad_inches=0.05)
plt.show()

# 9. Lifestyle/ECG statistical analysis


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

# =========================
# Example Data Setup
# =========================
# df should contain your ECG features + group labels
# Columns needed:
# - 'Group_MVPA' : "Active" / "Sedentary"
# - 'Group_BMI'  : "Normal" / "Overweight"
# - ECG features (columns in your significant_params list)

# Assume 'data' DataFrame is already loaded and contains necessary columns
# Ensure 'mvpa_category' column exists
data["mvpa_category"] = data["mvpa_minutes"].apply(lambda x: "Active" if x >= 150 else "Sedentary")
data["BMI_category"] = data["BMI"].apply(lambda x: "Normal" if x < 25 else "Overweight")


# significant ECG parameters
significant_params = [
     'Heart Rate', 'SDNN', 'RMSSD',
                     'pNN50',
                        'RMS Amp.', 'Mean R Amp.', 'R-Q Amp. Diff.',
                       'R-S Amp. Diff.',
                       'QRS Dur.', "BMI",
                     "mvpa_minutes"
]

# =========================
# Function to compute stats
# =========================
def compute_group_stats(df, feature, group_col, group1, group2):
    """Compute mean±SD, ΔMean, p-value, Cohen's d for two groups"""

    g1 = df[df[group_col] == group1][feature].dropna()
    g2 = df[df[group_col] == group2][feature].dropna()

    mean1, sd1 = g1.mean(), g1.std(ddof=1)
    mean2, sd2 = g2.mean(), g2.std(ddof=1)
    delta_mean = mean1 - mean2

    # t-test
    tstat, p = ttest_ind(g1, g2, equal_var=False)

    # Cohen's d
    pooled_sd = np.sqrt(((len(g1)-1)*sd1**2 + (len(g2)-1)*sd2**2) / (len(g1)+len(g2)-2))
    cohend = (mean1 - mean2) / pooled_sd if pooled_sd > 0 else np.nan

    return {
        "Feature": feature,
        f"{group1} (mean±SD)": f"{mean1:.2f} ± {sd1:.2f}",
        f"{group2} (mean±SD)": f"{mean2:.2f} ± {sd2:.2f}",
        "ΔMean": round(delta_mean, 2),
        "p": p,
        "Effect size (Cohen's d)": round(cohend, 2)
    }

# =========================
# Table A (Active vs Sedentary)
# =========================
results_A = []
for feat in significant_params:
    res = compute_group_stats(data, feat, "mvpa_category", "Active", "Sedentary")
    results_A.append(res)

table_A = pd.DataFrame(results_A)

# FDR correction
reject, pvals_adj, _, _ = multipletests(table_A["p"], method="fdr_bh")
table_A["p (FDR adj)"] = pvals_adj.round(5)
table_A.drop(columns=["p"], inplace=True)

print("\nTable A (ECG features: Active vs Sedentary):")
print(table_A)

# =========================
# Table B (Normal vs Overweight)
# =========================
results_B = []
for feat in significant_params:
    res = compute_group_stats(data, feat, "BMI_category", "Normal", "Overweight")
    results_B.append(res)

table_B = pd.DataFrame(results_B)

# FDR correction
reject, pvals_adj, _, _ = multipletests(table_B["p"], method="fdr_bh")
table_B["p (FDR adj)"] = pvals_adj.round(5)
table_B.drop(columns=["p"], inplace=True)

print("\nTable B (ECG features: Normal vs Overweight):")
print(table_B)

# =========================
# Optional: Save to CSV
# =========================
table_A.to_csv("/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/Table_A_ECG_MVPA.csv", index=False)
table_B.to_csv("/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/Table_B_ECG_BMI.csv", index=False)

In [ ]:
import pandas as pd
import math
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
import string

# =====================
# Font settings
# =====================
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 14,
    "axes.titlesize": 14,
    "axes.labelsize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14
})

# =====================
# Parameters
# =====================
significant_params = [
    'Heart Rate', 'SDNN', 'RMSSD',
    'pNN50', 'RMS Amp.', 'Mean R Amp.', 'R-Q Amp. Diff.',
    'R-S Amp. Diff.', 'QRS Dur.', "mvpa_minutes"
]

feature_name_map = {
    'Heart Rate': 'Heart Rate (bpm)',
    'SDNN': 'SDNN (ms)',
    'RMSSD': 'RMSSD (ms)',
    'pNN50': 'pNN50 (%)',
    'RMS Amp.': 'RMS Amp. (mV)',
    'Mean R Amp.': 'Mean R Amp. (mV)',
    'R-Q Amp. Diff.': 'R–Q Amp. Diff. (mV)',
    'R-S Amp. Diff.': 'R–S Amp. Diff. (mV)',
    'QRS Dur.': 'QRS Dur. (ms)',
    'BMI': 'BMI (kg/m$^2$)',
    'mvpa_minutes': 'MVPA (min/week)'
}

# =====================
# BOX PLOTS (5 per row)
# =====================
num_cols = 5
num_rows = math.ceil(len(significant_params) / num_cols)
letters = list(string.ascii_lowercase)  # ['a','b','c',...]

box_width = 0.3
#box_palette = ["#1f77b4", "crimson"]  # Active vs Sedentary

plt.figure(figsize=(12, 4.5))
for i, param in enumerate(significant_params):
    ax = plt.subplot(num_rows, num_cols, i + 1)
    sns.boxplot(
        x="mvpa_category",
        y=param,
        hue="mvpa_category",      # ✅ explicitly add hue
        dodge=False,              # ✅ keeps one box per category
        data=data,
        width=box_width,
        palette={"Active": "#1f77b4", "Sedentary": "crimson"},
        ax=ax
)
    # Safely remove legend if it exists
    if ax.legend_ is not None:
        ax.legend_.remove()

    # ✅ Remove "mvpa_category" axis label but keep Active/Sedentary ticks
    ax.set_xlabel("")
    ax.set_ylabel(feature_name_map.get(param, param))
    ax.set_title("")
    ax.tick_params(direction="in", length=3, width=1)

    # ✅ Add subplot label in top-right
    ax.text(
        0.95, 0.95, f"({letters[i]})",
        transform=ax.transAxes,
        ha="right", va="top",
        fontsize=14, fontweight="bold"
    )

plt.tight_layout()
plt.savefig("/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/ecg_features_with_activity_boxplots.pdf",format="pdf", bbox_inches="tight", pad_inches=0.05)
plt.savefig("/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/ecg_features_with_activity_boxplots.jpg",format="jpg", bbox_inches="tight", pad_inches=0.05)
plt.show()

# ================================
# Mann-Whitney U Test
# ================================
for param in significant_params:
    active_group = data[data["mvpa_category"] == "Active"][param]
    sedentary_group = data[data["mvpa_category"] == "Sedentary"][param]
    u_stat, p_value = mannwhitneyu(active_group, sedentary_group, alternative="two-sided")
    print(f"Mann-Whitney U Test for {param}: U-stat={u_stat:.3f}, p-value={p_value:.5f}")



In [ ]:
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 18,
    "axes.titlesize": 16,
    "axes.labelsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 18
})

# =========================
# SCATTER PLOTS (5 per row)
# =========================
num_cols_scatter = 5
num_rows_scatter = math.ceil(len(significant_params) / num_cols_scatter)

plt.figure(figsize=(18, 7))
for i, param in enumerate(significant_params):
    ax = plt.subplot(num_rows_scatter, num_cols_scatter, i + 1)
    sns.scatterplot(
        x="mvpa_minutes",
        y=param,
        hue="mvpa_category",
        data=data,
        palette={"Active": "#1f77b4", "Sedentary": "crimson"},
        edgecolor="black",
        alpha=0.8,
        s=80,
        ax=ax
    )
    sns.regplot(
        x="mvpa_minutes",
        y=param,
        data=data,
        scatter=False,
        line_kws={"color": "black", "linewidth": 2},
        ax=ax
    )
    ax.set_xlabel("MVPA (min/Week)", fontsize=16)
    ax.set_ylabel(feature_name_map.get(param, param), fontsize=18)
    ax.tick_params(direction="in", length=3, width=1)

    # ✅ Add subplot label in top-right
    ax.text(
        0.95, 0.95, f"({letters[i]})",
        transform=ax.transAxes,
        ha="right", va="top",
        fontsize=18, fontweight="bold"
    )

    # ✅ Show legend
    ax.legend( fontsize=16, frameon=True)

plt.tight_layout()
plt.savefig("/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/ecg_features_vs_mvpa_scatterplots.pdf",format="pdf", bbox_inches="tight", pad_inches=0.05)
plt.savefig("/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/ecg_features_vs_mvpa_scatterplots.jpg",format="jpg", bbox_inches="tight", pad_inches=0.05)
plt.show()

In [ ]:
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

results = []

for param in significant_params:
    X = data[["mvpa_minutes"]].values
    y = data[param].values

    # Fit linear regression
    model = LinearRegression().fit(X, y)
    coef = model.coef_[0]
    r2 = r2_score(y, model.predict(X))

    # Use statsmodels to get p-value
    X_sm = sm.add_constant(X)  # add intercept
    model_sm = sm.OLS(y, X_sm).fit()
    p_value = model_sm.pvalues[1]  # p-value for mvpa_minutes

    results.append({
        "Feature": feature_name_map.get(param, param),
        "Coefficient (β)": round(coef, 4),
        "R²": round(r2, 3),
        "p-value": f"{p_value:.6f}"
    })

# Create DataFrame
regression_table = pd.DataFrame(results)

# Show in notebook
print(regression_table)

# Save to Excel
regression_table.to_excel(
    "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/ecg_mvpa_regression_summary.xlsx",
    index=False
)

# Save LaTeX (optional)
with open("/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/ecg_mvpa_regression_summary.tex", "w") as f:
    f.write(regression_table.to_latex(index=False))



In [ ]:
import pandas as pd
import math
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
import string

# =====================
# Font settings
# =====================
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 14,
    "axes.titlesize": 14,
    "axes.labelsize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14
})

# =====================
# Parameters
# =====================
significant_params = [
    'Heart Rate', 'SDNN', 'RMSSD',
    'pNN50', 'RMS Amp.', 'Mean R Amp.', 'R-Q Amp. Diff.',
    'R-S Amp. Diff.', 'QRS Dur.', "BMI"
]

feature_name_map = {
    'Heart Rate': 'Heart Rate (bpm)',
    'SDNN': 'SDNN (ms)',
    'RMSSD': 'RMSSD (ms)',
    'pNN50': 'pNN50 (%)',
    'RMS Amp.': 'RMS Amp. (mV)',
    'Mean R Amp.': 'Mean R Amp. (mV)',
    'R-Q Amp. Diff.': 'R–Q Amp. Diff. (mV)',
    'R-S Amp. Diff.': 'R–S Amp. Diff. (mV)',
    'QRS Dur.': 'QRS Dur. (ms)',
    'BMI': 'BMI (kg/m$^2$)',
    'mvpa_minutes': 'MVPA (min/week)'
}

# =====================
# BOX PLOTS (5 per row)
# =====================
num_cols = 5
num_rows = math.ceil(len(significant_params) / num_cols)
letters = list(string.ascii_lowercase)  # ['a','b','c',...]

box_width = 0.3
#box_palette = ["#1f77b4", "crimson"]  # Active vs Sedentary

plt.figure(figsize=(14, 4.5))
for i, param in enumerate(significant_params):
    ax = plt.subplot(num_rows, num_cols, i + 1)
    sns.boxplot(
        x="BMI_category",
        y=param,
        hue="BMI_category",      # ✅ explicitly add hue
        dodge=False,              # ✅ keeps one box per category
        data=data,
        width=box_width,
        palette={"Normal": "#1f77b4", "Overweight": "crimson"},
        ax=ax
)
    # Safely remove legend if it exists
    if ax.legend_ is not None:
        ax.legend_.remove()

    # ✅ Remove "BMI_category" axis label but keep Active/Sedentary ticks
    ax.set_xlabel("")
    ax.set_ylabel(feature_name_map.get(param, param))
    ax.set_title("")
    ax.tick_params(direction="in", length=3, width=1)

    # ✅ Add subplot label in top-right
    ax.text(
        0.95, 0.95, f"({letters[i]})",
        transform=ax.transAxes,
        ha="right", va="top",
        fontsize=14, fontweight="bold"
    )

plt.tight_layout()
plt.savefig("/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/ecg_features_with_bmi_boxplots.pdf",format="pdf", bbox_inches="tight", pad_inches=0.05)
plt.savefig("/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/ecg_features_with_bmi_boxplots.jpg",format="jpg", bbox_inches="tight", pad_inches=0.05)
plt.show()
# ================================
# Mann-Whitney U Test
# ================================
# Mann-Whitney U Test between Active vs Sedentary for each parameter
for param in significant_params:
    normal_group = data[data["BMI_category"] == "Normal"][param]
    overweight_group = data[data["BMI_category"] == "Overweight"][param]
    u_stat, p_value = mannwhitneyu(normal_group, overweight_group, alternative="two-sided")
    print(f"Mann-Whitney U Test for {param}: U-stat={u_stat:.3f}, p-value={p_value:.5f}")

In [ ]:
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 18,
    "axes.titlesize": 16,
    "axes.labelsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 18
})

# =========================
# SCATTER PLOTS (5 per row)
# =========================
num_cols_scatter = 5
num_rows_scatter = math.ceil(len(significant_params) / num_cols_scatter)

plt.figure(figsize=(18, 7))
for i, param in enumerate(significant_params):
    ax = plt.subplot(num_rows_scatter, num_cols_scatter, i + 1)
    sns.scatterplot(
        x="BMI",
        y=param,
        hue="BMI_category",
        data=data,
        palette={"Normal": "#1f77b4", "Overweight": "crimson"},
        edgecolor="black",
        alpha=0.8,
        s=80,
        ax=ax
    )
    sns.regplot(
        x="BMI",
        y=param,
        data=data,
        scatter=False,
        line_kws={"color": "black", "linewidth": 2},
        ax=ax
    )
    ax.set_xlabel("BMI (kg/m$^2$)", fontsize=16)
    ax.set_ylabel(feature_name_map.get(param, param), fontsize=18)
    ax.tick_params(direction="in", length=3, width=1)

    # ✅ Add subplot label in top-right
    ax.text(
        0.95, 0.95, f"({letters[i]})",
        transform=ax.transAxes,
        ha="right", va="top",
        fontsize=18, fontweight="bold"
    )

    # ✅ Show legend
    ax.legend( fontsize=16, frameon=True)

plt.tight_layout()
plt.savefig("/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/ecg_features_vs_bmi_scatterplots.pdf",format="pdf", bbox_inches="tight", pad_inches=0.05)
plt.savefig("/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/ecg_features_vs_bmi_scatterplots.jpg",format="jpg", bbox_inches="tight", pad_inches=0.05)
plt.show()

In [ ]:
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

results_bmi = []

for param in significant_params:
    X = data[["BMI"]].values
    y = data[param].values

    # Fit linear regression
    model = LinearRegression().fit(X, y)
    coef = model.coef_[0]
    r2 = r2_score(y, model.predict(X))

    # Use statsmodels to get p-value
    X_sm = sm.add_constant(X)  # add intercept
    model_sm = sm.OLS(y, X_sm).fit()
    p_value = model_sm.pvalues[1]  # p-value for BMI coefficient

    results_bmi.append({
        "Feature": feature_name_map.get(param, param),
        "Coefficient (β)": round(coef, 4),
        "R²": round(r2, 3),
        "p-value": f"{p_value:.6f}"
    })

# Create DataFrame
regression_table_bmi = pd.DataFrame(results_bmi)

# Show in notebook
print(regression_table_bmi)

# Save to Excel
regression_table_bmi.to_excel(
    "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/ecg_bmi_regression_summary.xlsx",
    index=False
)

# Save LaTeX (optional)
with open("/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/ecg_bmi_regression_summary.tex", "w") as f:
    f.write(regression_table_bmi.to_latex(index=False))


In [ ]:
significant_params = [
    'Heart Rate', 'SDNN', 'RMSSD',
    'pNN50', 'RMS Amp.', 'Mean R Amp.', 'R-Q Amp. Diff.',
    'R-S Amp. Diff.', 'QRS Dur.', "mvpa_minutes","BMI"
]
feature_name_map = {
    'Heart Rate': 'Heart Rate (bpm)',
    'SDNN': 'SDNN (ms)',
    'RMSSD': 'RMSSD (ms)',
    'pNN50': 'pNN50 (%)',
    'RMS Amp.': 'RMS Amp. (mV)',
    'Mean R Amp.': 'Mean R Amp. (mV)',
    'R-Q Amp. Diff.': 'R–Q Amp. Diff. (mV)',
    'R-S Amp. Diff.': 'R–S Amp. Diff. (mV)',
    'QRS Dur.': 'QRS Dur. (ms)',
    'BMI': 'BMI (kg/m$^2$)',
    'mvpa_minutes': 'MVPA (min/week)'
}
# Select the significant parameters from the DataFrame
significant_data = data[significant_params]

# Compute the covariance matrix for the significant parameters
cov_matrix = significant_data.cov()
cov_matrix.rename(index=feature_name_map, columns=feature_name_map, inplace=True)
# Plot the covariance matrix using seaborn
plt.figure(figsize=(10, 6))  # Adjust figure size as needed
sns.heatmap(cov_matrix, annot=True, cmap='RdBu_r', fmt='.2f', linewidths=0.5)

#plt.title("Covariance Matrix of Significant Parameters")
plt.xticks(rotation=90, ha='right')  # Rotate x-axis labels for better readability
plt.yticks(rotation=0)  # Keep y-axis labels horizontal
plt.tick_params(axis='both', direction='in', length=3, width=1)
plt.tight_layout()  # Adjust layout to prevent labels from overlapping
plt.savefig(
    "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/significant_param_covariance.pdf",
    format="pdf", bbox_inches="tight", pad_inches=0.05
)
plt.show()

In [ ]:
# 4. Heatmap for correlations of significant parameters
correlation_matrix = data[significant_params].corr()

# Apply feature_name_map to both rows and columns
correlation_matrix.rename(index=feature_name_map, columns=feature_name_map, inplace=True)

plt.figure(figsize=(10, 6))
sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="RdBu_r",
    fmt=".2f"
      # optional: makes colorbar smaller
)

#plt.title("Correlation Heatmap of Significant Parameters", fontsize=14)
plt.xticks(rotation=90, ha="right", fontsize=12)
plt.yticks(rotation=0, fontsize=12)

# Tick marks inside
plt.tick_params(axis='both', direction='in', length=3, width=1)

plt.tight_layout()
plt.savefig(
    "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/significant_param_correlation.pdf",
    format="pdf", bbox_inches="tight", pad_inches=0.05
)
plt.savefig(
    "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/significant_param_correlation.jpg",
    format="jpg", bbox_inches="tight", pad_inches=0.05
)
plt.show()


# 10. Diet, sleep and categorical lifestyle analyses


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import mannwhitneyu, ttest_ind, shapiro, chi2_contingency
from math import sqrt
import os

# =========================
# Load data
# =========================
file_path = "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/cluster_assignments.csv"
df = pd.read_csv(file_path)

diet_vars = ["processed_meats", "sugary_foods", "fruits_vegetables"]

# Output folder for plots
out_dir = "/content/drive/MyDrive/ECG New/Lifestyle data/Lifestyle_Figures"
os.makedirs(out_dir, exist_ok=True)

# =========================
# Helper functions
# =========================
def cohens_d(x, y):
    nx, ny = len(x), len(y)
    dof = nx + ny - 2
    pooled_std = sqrt(((nx - 1)*x.std()**2 + (ny - 1)*y.std()**2) / dof)
    return (x.mean() - y.mean()) / pooled_std

def cliffs_delta(x, y):
    n, m = len(x), len(y)
    more = sum(i > j for i in x for j in y)
    less = sum(i < j for i in x for j in y)
    return (more - less) / (n * m)

def cramers_v(confusion_matrix):
    chi2, _, _, _ = chi2_contingency(confusion_matrix)
    n = confusion_matrix.sum().sum()
    r, k = confusion_matrix.shape
    return sqrt(chi2 / (n * (min(r-1, k-1))))

# =========================
# 1. Dietary Habits
# =========================
results = []

for var in diet_vars:
    cluster0 = df[df["Cluster"]==0][var].dropna()
    cluster1 = df[df["Cluster"]==1][var].dropna()

    # Normality check
    p0, p1 = shapiro(cluster0).pvalue, shapiro(cluster1).pvalue
    normal = (p0 > 0.05) and (p1 > 0.05)

    if normal:
        stat, p = ttest_ind(cluster0, cluster1, equal_var=False)
        eff = cohens_d(cluster0, cluster1)
        eff_name = "Cohen's d"
    else:
        stat, p = mannwhitneyu(cluster0, cluster1, alternative='two-sided')
        eff = cliffs_delta(cluster0, cluster1)
        eff_name = "Cliff's Δ"

    results.append({
        "Variable": var.replace("_"," ").title(),
        "Cluster0 Mean±SD": f"{cluster0.mean():.2f} ± {cluster0.std():.2f}",
        "Cluster1 Mean±SD": f"{cluster1.mean():.2f} ± {cluster1.std():.2f}",
        "Test": "t-test" if normal else "Mann–Whitney U",
        "p-value": f"{p:.4f}",
        "Effect Size": f"{eff:.3f} ({eff_name})"
    })

# =========================
# Subplot for Dietary Habits with custom labels
# =========================
custom_colors = ["#1f77b4", "crimson"] # Define custom_colors before use

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5), sharey=False)

ylabels = [
    "Processed Meats Score (0–10)",
    "Sugary Foods Score (0–10)",
    "Fruits & Vegetables Score (0–10)"
]

for i, (var, ylabel) in enumerate(zip(diet_vars, ylabels)):
    ax = axes[i]
    sns.violinplot(
        data=df, x="Cluster", y=var, inner=None,
        palette=[custom_colors[0], custom_colors[1]], ax=ax
    )
    sns.swarmplot(
        data=df, x="Cluster", y=var, color="k", alpha=0.6, ax=ax
    )

    # Descriptive y-axis
    ax.set_ylabel(ylabel, fontsize=14)
    ax.set_xlabel("Cluster", fontsize=14)

    # Remove default title
    ax.set_title("")
    ax.tick_params(axis='both', direction='in', length=3, width=1)
    # Add subplot label (a), (b), (c) inside top-right
    ax.text(
        0.98, 0.98, f"({chr(97+i)})",
        transform=ax.transAxes, ha="right", va="top",
        fontsize=14, fontweight="bold"
    )

plt.tight_layout()

plt.savefig(f"{out_dir}/diet_by_cluster_subplots.pdf", format="pdf", bbox_inches="tight", pad_inches=0.05)
plt.savefig(f"{out_dir}/diet_by_cluster_subplots.jpg", format="jpg", bbox_inches="tight", pad_inches=0.05)
plt.show()


diet_table = pd.DataFrame(results)
print("\n=== Dietary Habits Results ===")
print(diet_table.to_string(index=False))

# =========================
# 2. Sleep Quality
# =========================
df["SleepCategory"] = np.where(df["sleep_quality"] >= 7, "Good", "Poor")

cont_table = pd.crosstab(df["Cluster"], df["SleepCategory"])
chi2, p, dof, expected = chi2_contingency(cont_table)
v = cramers_v(cont_table)
print("Contingency Table:\n", cont_table)
sleep_table = pd.DataFrame({
    "Good Sleep (%)": (cont_table["Good"]/cont_table.sum(axis=1)*100).round(1),
    "Poor Sleep (%)": (cont_table["Poor"]/cont_table.sum(axis=1)*100).round(1),
})
sleep_table["Chi-square p"] = f"{p:.4f}"
sleep_table["Cramer’s V"] = f"{v:.3f}"


sleep_table["Chi-square p"] = f"{p:.4f}"
sleep_table["Cramer’s V"] = f"{v:.3f}"

print("\n=== Sleep Quality Results ===")
print(sleep_table.to_string())

print("\n=== Sleep Quality Chi-square Test ===")
print(f"Chi² = {chi2:.3f}, df = {dof}, p = {p:.4f}, Cramer’s V = {v:.3f}")
# === Sleep Figure ===
cont_table_pct = cont_table.div(cont_table.sum(axis=1), axis=0)

# === Sleep Figure with custom colors ===
# custom_colors = {0: "#1f77b4", 1: "#ff7f0e"}  # blue & orange
custom_colors = {"Good": "#1f77b4", "Poor": "crimson"} # blue & orange
cont_table_pct = cont_table.div(cont_table.sum(axis=1), axis=0)
plt.figure(figsize=(4,3))
ax = cont_table_pct.plot(
    kind="bar", stacked=True, figsize=(5,4),
    color=[custom_colors[c] for c in cont_table_pct.columns]
)
plt.title("Sleep Categories Across Clusters", fontsize=12)
plt.ylabel("Percentage (%)", fontsize=12)
plt.xlabel("Cluster")
plt.legend(loc="upper right", fontsize=11)
plt.tight_layout()
plt.tick_params(axis='both', direction='in', length=3, width=1)
plt.savefig(f"{out_dir}/sleep_quality_by_cluster.pdf", format="pdf")
plt.show()
plt.close()


plt.figure(figsize=(4,3))
sns.boxplot(
    x="Cluster", y="sleep_quality",
    hue="SleepCategory",   # Good / Poor
    data=df,
    palette={"Good": "#1f77b4", "Poor": "crimson"}
)

plt.xlabel("Cluster", fontsize=12)
plt.ylabel("Sleep Quality Score (0–10)", fontsize=12)
plt.legend(title="Sleep Category", title_fontsize=12, fontsize=12)

plt.tick_params(axis='both', direction='in', length=3, width=1)
plt.tight_layout()
plt.savefig(f"{out_dir}/sleep_quality_boxplot.pdf", format="pdf", bbox_inches="tight", pad_inches=0.05)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Set font globally
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 14
})

# Custom colors
mvpa_colors = {"Active": "#1f77b4", "Sedentary": "crimson"}
bmi_colors  = {"Normal": "#1f77b4", "Overweight": "crimson"}
sleep_colors = {"Good": "#1f77b4", "Poor": "crimson"}

# Bar width
bar_width = 0.25

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.5))  # 1 row, 3 columns

# =======================
# a) MVPA
# =======================
mvpa_pct.plot(
    kind="bar", stacked=True, color=[mvpa_colors[c] for c in mvpa_pct.columns],
    ax=axes[0], width=bar_width
)
axes[0].set_xlabel("Cluster", fontsize=14)
axes[0].set_ylabel("Percentage (%)", fontsize=14)
axes[0].set_title("MVPA Categories Across Clusters", fontsize=14)
axes[0].text(0.95, 0.95, "(a)", transform=axes[0].transAxes,
             ha="right", va="top", fontsize=14, fontweight="bold")
axes[0].tick_params(axis='both', direction='in', length=3, width=1)
axes[0].legend( fontsize=12, title_fontsize=12)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)  # upright

# =======================
# b) BMI
# =======================
bmi_pct.plot(
    kind="bar", stacked=True, color=[bmi_colors[c] for c in bmi_pct.columns],
    ax=axes[1], width=bar_width
)
axes[1].set_xlabel("Cluster", fontsize=14)
axes[1].set_ylabel("Percentage (%)", fontsize=14)
axes[1].set_title("BMI Categories Across Clusters", fontsize=14)
axes[1].text(0.95, 0.95, "(b)", transform=axes[1].transAxes,
             ha="right", va="top", fontsize=14, fontweight="bold")
axes[1].tick_params(axis='both', direction='in', length=3, width=1)
axes[1].legend(fontsize=12, title_fontsize=12)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)  # upright

# =======================
# c) Sleep
# =======================
cont_table_pct.plot(
    kind="bar", stacked=True, color=[sleep_colors[c] for c in cont_table_pct.columns],
    ax=axes[2], width=bar_width
)
axes[2].set_xlabel("Cluster", fontsize=14)
axes[2].set_ylabel("Percentage (%)", fontsize=14)
axes[2].set_title("Sleep Categories Across Clusters", fontsize=14)
axes[2].text(0.95, 0.95, "(c)", transform=axes[2].transAxes,
             ha="right", va="top", fontsize=14, fontweight="bold")
axes[2].tick_params(axis='both', direction='in', length=3, width=1)
axes[2].legend( fontsize=12, title_fontsize=12)
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=0)  # upright

plt.tight_layout()
plt.savefig(f"{out_dir}/lifestyle_categories_stacked_subplots_individual_legend.pdf",
            format="pdf", bbox_inches="tight", pad_inches=0.05)
plt.savefig(f"{out_dir}/lifestyle_categories_stacked_subplots_individual_legend.jpg",
            format="jpg", bbox_inches="tight", pad_inches=0.05)
plt.show()



# 11. Optimal cluster-number diagnostics


In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# Assuming 'scaled_features' is already defined and contains your scaled data

# Determine the range of k values to test
max_k = 10
wcss = []

# Calculate WCSS for each k
for k in range(1, max_k + 1):
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(scaled_features)
    wcss.append(kmeans.inertia_) # inertia_ is the WCSS

# Plot the elbow method graph
plt.figure(figsize=(8, 4))
plt.plot(range(1, max_k + 1), wcss, marker='o', linestyle='--', color='green')
#plt.title('Elbow Method for Optimal K', fontsize=14)
plt.xlabel('Number of Clusters (k)', fontsize=12)
plt.ylabel('Within-Cluster Sum of Squares (WCSS)', fontsize=12)
plt.xticks(range(1, max_k + 1))
plt.tick_params(axis='both', direction='in', length=3, width=1)
plt.grid(True, linestyle=':', alpha=0.6)
plt.savefig(
    "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/Elbow.pdf",
    format="pdf", bbox_inches="tight", pad_inches=0.05
)
plt.savefig(
    "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/Elbow.jpg",
    format="jpg", bbox_inches="tight", pad_inches=0.05
)
plt.show()

print("The elbow method plot shows the WCSS for each number of clusters (k). Look for the 'elbow' point where the decrease in WCSS starts to level off. This point is often considered the optimal number of clusters.")

# Based on the plot, you can then choose the optimal k and run KMeans again
# For example, if the elbow appears to be at k=2:
# optimal_k = 2
# kmeans_final = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
# clusters = kmeans_final.fit_predict(scaled_features)
# data['Cluster'] = clusters # Assign the new cluster labels to your DataFrame

In [ ]:
# Silhouette Score for confirmation
# ====================================
import numpy as np
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans # Import KMeans

# Define the range of k values to test
max_k = 10 # Assuming the same max_k as the elbow method
K_range = range(1, max_k + 1)

silhouette_scores = []
# Assuming 'scaled_features' is already defined and contains your scaled data
# Assuming X_scaled refers to scaled_features based on previous code
X_scaled = scaled_features

for k in K_range:
    if k == 1: # Skip silhouette score calculation for k=1
        silhouette_scores.append(np.nan) # Append NaN or some placeholder for k=1
        continue
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_scaled)
    silhouette_avg = silhouette_score(X_scaled, cluster_labels)
    silhouette_scores.append(silhouette_avg)

# Plot Silhouette scores
plt.figure(figsize=(6, 4))
plt.plot(K_range, silhouette_scores, 'o--', color='green')
plt.xlabel("Number of Clusters (k)", fontsize=12)
plt.ylabel("Silhouette Score", fontsize=12)
#plt.title("Silhouette Score for Different k", fontsize=14)

plt.xticks(K_range)
plt.tick_params(axis='both', direction='in', length=3, width=1)
plt.grid(True, linestyle=':', alpha=0.6)
plt.savefig(
    "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/silhoutte_score_optimal_cluster.pdf",
    format="pdf", bbox_inches="tight", pad_inches=0.05
)
plt.savefig(
    "/content/drive/MyDrive/ECG New/Data_new/Filtered_data_new/silhoutte_score_optimal_cluster.jpg",
    format="jpg", bbox_inches="tight", pad_inches=0.05
)
#plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

# ====================================
# Print best k suggestion
# ====================================
# Find the index of the max silhouette score, starting from the second element (k=2)
best_k_index = np.nanargmax(silhouette_scores) # Use nanargmax to handle NaN at index 0
best_k = K_range[best_k_index]


print(f"✅ Based on Silhouette Score, the optimal number of clusters is: {best_k}")